# Vehicle Insurance Cross-Sell Response Propensity — Tier 0 Experiments

## Notebook purpose and status

This notebook is a clean Tier 0 experiment for the vehicle-insurance cross-sell response-propensity project. It is intentionally separate from the original `experimentation.ipynb`, which is preserved as a historical baseline.

The purpose of this notebook is to establish a trustworthy evaluation protocol before adding advanced feature engineering, resampling, or additional model families. Results from this notebook should be reproducible from top to bottom and should not depend on hidden state from the previous notebook.

**Current phase:** Tier 0 — evaluation correctness and business-aligned baseline measurement.

**Important status note:** this notebook is an experimental research artifact. It is not yet the final training pipeline or the production serving implementation.

## 1. Project objective

The project estimates which existing customers are most likely to respond positively to a vehicle-insurance cross-sell offer. The model receives customer information available before a marketing campaign and produces a response-propensity score. Customers are then ranked from highest to lowest score so that a campaign team can prioritize a selected top-K group.

The target is `Response`:

- `Response = 1`: the customer responded positively to the vehicle-insurance offer.
- `Response = 0`: the customer did not respond positively to the vehicle-insurance offer.

The current Phase 1 business action is **top-K customer selection**. The company may select K as a percentage of the scored population or as a fixed number of customers. This notebook evaluates top 1%, 5%, 10%, and 20% capacities, with top 10% as the initial illustrative operating point. Customers outside the selected group are lower-priority campaign cases; they are not rejected, denied coverage, or declared ineligible.

This model is a marketing-prioritization model only. It must not be used for claims approval, premium pricing, underwriting, fraud detection, insurance eligibility, policy cancellation, or automatic adverse decisions. The dataset supports response-propensity modeling, not causal claims about whether contacting a customer created an incremental response.

## 2. Evaluation contract

The primary question is:

> Does the model rank likely responders near the top well enough to improve the quality of a fixed-capacity marketing selection?

### Primary metrics

The primary metrics are ranking and top-K selection metrics:

- **PR-AUC / Average Precision:** measures ranking quality with greater focus on the minority positive class.
- **Precision@K:** among the customers selected in the top K%, how many actually responded?
- **Recall@K:** among all actual responders, how many were captured in the top K%?
- **Lift@K:** how much better the top-K selection performs compared with selecting customers at random.

Calculate these metrics at top 1%, 5%, 10%, and 20%. The operating point must be selected using validation data, not the final test set.

### Secondary metrics

The notebook may also report ROC-AUC, precision, recall, F1, and a confusion matrix. These are secondary because F1 and confusion-matrix metrics depend on a classification threshold, while the campaign decision is primarily based on ranking. The default threshold of 0.5 must not be treated as automatically correct.

### Data split and leakage rules

Use a stratified three-way split with training, validation, and one final untouched test set. The current recommended protocol is 80% training, 10% validation, and 10% final test, with a fixed random seed of 42 and cross-validation performed only inside the training portion when hyperparameters are tuned.

- Fit preprocessing only on training data.
- Fit models and perform hyperparameter search using training data.
- Use validation data for model comparisons, top-K operating-point decisions, and threshold analysis.
- Keep the final test set untouched until the model and evaluation protocol are frozen.
- Evaluate the final selected model on the final test set once for the final report.
- Never select a model because it produced the best final-test score.
- Compare all candidate models using the same split, preprocessing rules, and metric definitions.

### Baseline and model-selection rules

The Tier 0 comparison should include a simple baseline, an interpretable baseline where possible, and the current no-resampling LightGBM baseline. The purpose is to establish whether the model provides useful ranking skill above simple alternatives before introducing more complex changes.

A candidate should be judged primarily on PR-AUC and top-K selection quality, especially the agreed top-K capacities. Threshold-based metrics can inform the serving decision, but they must not replace the ranking evaluation.

## 3. Tier 0 notebook workflow

The notebook should follow this order:

1. Document the project objective and evaluation contract.
2. Load the expected dataset using a portable path.
3. Run dataset integrity checks and record the dataset snapshot.
4. Create the frozen train/validation/final-test split.
5. Build the baseline preprocessing and modeling flow.
6. Generate validation probabilities and calculate PR-AUC and top-K metrics.
7. Compare the baseline candidates using validation data only.
8. Perform threshold analysis as a secondary evaluation.
9. Freeze the final candidate and operating point.
10. Evaluate the frozen candidate once on the final test set.
11. Summarize findings, limitations, and the decision about whether Tier 1 is justified.

Do not move to advanced feature engineering or resampling until the Tier 0 baseline and evaluation protocol are working end to end.

## Notebook dependencies

In [1]:
import os
import warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import SMOTE

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    auc,
    classification_report,
    roc_auc_score
)

import lightgbm as lgb
from lightgbm import LGBMClassifier
import mlflow
import pickle

warnings.filterwarnings("ignore")

## 4. Dataset loading and integrity checks

This section establishes that the notebook is using the expected dataset and that the data is safe to use for modeling. It is not intended to repeat every exploratory chart from the original notebook. The checks here are reproducibility and data-integrity gates that should run whenever this notebook is executed.

### Dataset-loading responsibilities

The loading step should document the expected file location, load the dataset into a dataframe, and record the dataset shape and columns. The path should be portable rather than tied to one absolute directory on one computer. The notebook should make it clear which data snapshot produced the experiment results.

### Integrity checks to complete

The code cells below should verify the following items before feature preparation or splitting:

1. **Expected schema:** confirm that the expected feature columns and the `Response` target exist.
2. **Unexpected columns:** identify columns that are not part of the documented data contract.
3. **Target validity:** confirm that `Response` contains only 0 and 1 and record the positive-class prevalence.
4. **Missing values:** report missing values by column and confirm whether any handling is required.
5. **Duplicate records:** check duplicate IDs and complete duplicate rows.
6. **Identifier handling:** confirm that `id` is unique, is not used as a predictive feature, and can be retained separately for future ranked-customer output.
7. **Data types:** confirm that numeric columns are numeric and that categorical-looking columns such as `Gender`, `Vehicle_Age`, `Vehicle_Damage`, `Region_Code`, and `Policy_Sales_Channel` are treated according to their meaning rather than only their storage dtype.
8. **Value ranges:** check that fields such as `Age`, `Annual_Premium`, `Vintage`, `Driving_License`, and `Previously_Insured` contain plausible values.
9. **Leakage review:** confirm that no target, post-campaign, or future-information column is included among the features.
10. **Dataset limitations:** record that the current dataset has no reliable campaign timestamp, is treated as one row per customer, and does not contain treatment/control information for causal or uplift modeling.

### Expected output

At the end of this section, the notebook should provide a concise dataset summary containing the row count, column list, target counts and prevalence, missing-value summary, duplicate summary, identifier check, and any accepted limitations. A reviewer should be able to understand what data entered the experiment without opening the old notebook.

In [2]:
from pathlib import Path

# Support execution from either the repository root or the notebooks directory.
current_dir = Path.cwd()
data_path = (current_dir / "data.csv") if current_dir.name == "notebooks" else (current_dir / "notebooks" / "data.csv")
data_path = data_path.resolve()

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Dataset shape: {df.shape}")
df.head()

Loaded dataset from: D:\projVehicleInsurance\vehicle-Insurance\notebooks\data.csv
Dataset shape: (381109, 12)


,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


confirms that no unexpected columns exist and schema is as expected. 

In [4]:
response_counts = df['Response'].value_counts()
print(f"Response counts:\n{response_counts}\n")

Response counts:
Response
0    334399
1     46710
Name: count, dtype: int64



In [5]:
positive_percentage = (response_counts[1] / response_counts.sum()) * 100
print(f"Percentage of positive responses: {positive_percentage:.2f}%")

Percentage of positive responses: 12.26%


The response counts confirm that response contains only 0 and 1 and the positive-class prevalence is 12.26%.


In [6]:
#Lets now check for missing values in each column
df.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

Thus no missing values exist in any column.

In [7]:
#lets now check for duplicate ids and rows in the dataset
duplicate_ids = df.duplicated(subset='id').sum()
print(f"Number of duplicate IDs: {duplicate_ids}")

duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate IDs: 0
Number of duplicate rows: 0


So no duplicate IDs or rows exist in the dataset.

In [8]:
#check for categorical columns and their unique values
categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    print(f"\nUnique values in column '{col}':")
    print(df[col].unique())


Unique values in column 'Gender':
['Male' 'Female']

Unique values in column 'Vehicle_Age':
['> 2 Years' '1-2 Year' '< 1 Year']

Unique values in column 'Vehicle_Damage':
['Yes' 'No']


In [9]:
# Check documented value ranges and binary domains.
range_violations = {
    "Age_non_positive": int((df["Age"] <= 0).sum()),
    "Annual_Premium_non_positive": int((df["Annual_Premium"] <= 0).sum()),
    "Vintage_negative": int((df["Vintage"] < 0).sum()),
    "Driving_License_invalid": int((~df["Driving_License"].isin([0, 1])).sum()),
    "Previously_Insured_invalid": int((~df["Previously_Insured"].isin([0, 1])).sum()),
}
range_violations = {name: count for name, count in range_violations.items() if count > 0}
if range_violations:
    raise ValueError(f"Value-range violations found: {range_violations}")


In [10]:
print("Dataset integrity checks passed.")

Dataset integrity checks passed.


## 5. Data Preparation

Lets prepare the data now

In [11]:
random_state = 42

In [12]:
X = df.drop(columns=["Response", "id"])
y = df["Response"]
id_col = df["id"]
print(f"Features shape: {X.shape}, Target shape: {y.shape}")

Features shape: (381109, 10), Target shape: (381109,)


In [13]:
# LETS PREPARE THE TRAIN,TEST AND VALIDATION DATASETS
#first lets separate train  and rest in 80:20 ratio
X_train, X_rest, y_train, y_rest, id_train, id_rest = train_test_split(X, y, id_col, test_size=0.2, random_state=random_state, stratify=y)

#now divide rest into test and validation in 50:50 ratio
X_test, X_val, y_test, y_val, id_test, id_val = train_test_split(X_rest, y_rest, id_rest, test_size=0.5, random_state=random_state, stratify=y_rest)


In [14]:
print("Training set:")
print(X_train.shape, y_train.shape)

print("\nValidation set:")
print(X_val.shape, y_val.shape)

print("\nTest set:")
print(X_test.shape, y_test.shape)

Training set:
(304887, 10) (304887,)

Validation set:
(38111, 10) (38111,)

Test set:
(38111, 10) (38111,)


In [15]:
# Verify target prevalence, ID separation, and complete dataset coverage.
split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test)],
    "positive_responses": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

print("Target prevalence by split:")
display(split_summary.style.format({"positive_rate": "{:.4%}"}))

# Every pair of subsets must contain different customer IDs.
train_ids = set(id_train)
validation_ids = set(id_val)
test_ids = set(id_test)

overlap_counts = {
    "train_validation": len(train_ids & validation_ids),
    "train_test": len(train_ids & test_ids),
    "validation_test": len(validation_ids & test_ids),
}

if any(overlap_counts.values()):
    raise ValueError(f"Customer ID overlap detected: {overlap_counts}")

# The split must cover every original row exactly once.
combined_ids = train_ids | validation_ids | test_ids
expected_ids = set(id_col)
missing_ids = expected_ids - combined_ids
unexpected_ids = combined_ids - expected_ids
total_split_rows = len(y_train) + len(y_val) + len(y_test)

if missing_ids or unexpected_ids or total_split_rows != len(df):
    raise ValueError(
        "Dataset coverage check failed: "
        f"missing_ids={len(missing_ids)}, "
        f"unexpected_ids={len(unexpected_ids)}, "
        f"split_rows={total_split_rows}, original_rows={len(df)}"
    )

print("ID overlap check passed: no customer appears in more than one split.")
print("Coverage check passed: every original customer appears exactly once across the splits.")

Target prevalence by split:


,rows,positive_responses,positive_rate
train,304887,37368,12.2563%
validation,38111,4671,12.2563%
test,38111,4671,12.2563%


ID overlap check passed: no customer appears in more than one split.
Coverage check passed: every original customer appears exactly once across the splits.


## 6. Baseline Preprocessing and Modeling Flow

### 6.1 Preprocessing

In [16]:
# lets firts define the feature types
categorical_features = [
    "Gender",
    "Vehicle_Age",
    "Vehicle_Damage",
    "Region_Code",
    "Policy_Sales_Channel",
]
numerical_features = ["Age", "Annual_Premium", "Vintage"]
binary_features = ["Driving_License", "Previously_Insured"]

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", "passthrough", numerical_features),
        ("binary", "passthrough", binary_features)
    ]
)

In [18]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

In [19]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_val_processed = preprocessor.transform(X_val)

Lets check the preprocessed data

In [20]:
print("Train columns:", X_train_processed.shape[1])
print("Validation columns:", X_val_processed.shape[1])
print("Test columns:", X_test_processed.shape[1])

Train columns: 219
Validation columns: 219
Test columns: 219


In [21]:
feature_names = preprocessor.get_feature_names_out()

In [22]:
same_feature_structure = (
    X_train_processed.shape[1] == len(feature_names)
    and X_val_processed.shape[1] == len(feature_names)
    and X_test_processed.shape[1] == len(feature_names)
)
print("Same feature structure:", same_feature_structure)

Same feature structure: True


In [23]:
numerical_present = all(
    f"numerical__{col}" in feature_names
    for col in numerical_features
)

# 4. Binary features included
binary_present = all(
    f"binary__{col}" in feature_names
    for col in binary_features
)


print("All numerical features included:", numerical_present)
print("All binary features included:", binary_present)

All numerical features included: True
All binary features included: True


In [24]:
target_present = any("Response" in col for col in feature_names)

id_present = any(
    col == "id" or col.endswith("__id")
    for col in feature_names
)
print("Response present:", target_present)
print("ID present:", id_present)

Response present: False
ID present: False


### 6.2 Define metrics

In [25]:
def precision_at_k(y_true , y_proba , k):
    """
    Calculate precision at k% for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider.

    Returns:
    float: Precision at k.
    """
    # Ensure inputs are numpy arrays
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    n_top = int(len(y_true) * (k / 100))
    
    y_top_k_indices = np.argsort(y_proba)[::-1][:n_top]
    top_k_true = y_true[y_top_k_indices]
    
    return top_k_true.sum() / n_top if n_top > 0 else 0.0

    

In [26]:
def recall_at_k(y_true , y_proba , k):
    """
    Calculate recall at k% for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider.

    Returns:
    float: Recall at k.
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    n_top = int(len(y_true) * (k / 100))
    top_k_indices = np.argsort(y_proba)[::-1][:n_top]
    top_k_true = y_true[top_k_indices]
    
    return top_k_true.sum() / y_true.sum() if y_true.sum() > 0 else 0.0

In [27]:
def Lift_at_k(y_true,y_proba,k):
    """
    Calculate lift at k% for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider.

    Returns:
    float: Lift at k.
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    precision_value = precision_at_k(y_true, y_proba, k)
    overall_positive_rate = np.mean(y_true)

    return precision_value / overall_positive_rate if overall_positive_rate > 0 else 0.0 

In [28]:
def pr_auc(y_true, y_proba):
    """
    Calculate the area under the precision-recall curve (PR AUC) for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.

    Returns:
    float: Area under the precision-recall curve.
    """
    return average_precision_score(y_true, y_proba)

### 6.3 MLFLOW

Before Training lets initiate the mlflow

In [29]:
def metric_logger(y_true,y_pred, y_proba, k):
    """
    Log various evaluation metrics for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_pred (array-like): Predicted binary labels.
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider for precision, recall, and lift.

    Returns:
    dict: Dictionary containing calculated metrics.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        
        "precision_at_k": precision_at_k(y_true, y_proba, k),
        "recall_at_k": recall_at_k(y_true, y_proba, k),
        "lift_at_k": Lift_at_k(y_true, y_proba, k),
        "pr_auc": pr_auc(y_true, y_proba)
    }
    mlflow.log_metrics(metrics)
    return metrics

In [30]:
from dotenv import load_dotenv
load_dotenv()  

True

In [31]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if not tracking_uri:
    raise ValueError("MLFLOW_TRACKING_URI is not set in the environment variables.")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Vehicle_Insurance_Experimentation_version2")
print(f"Tracking URI: {tracking_uri}")
print("MLflow experiment configured.")

Tracking URI: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow
MLflow experiment configured.


### 6.4 Model Training

train with lightgbm.

In [33]:
#MLFLOW RUN
run_name = "run0_lgbm_no_resampling_baseline"
with mlflow.start_run(run_name=run_name) as run:
    print(f"MLflow run started with ID: {run.info.run_id}")
    mlflow.log_params(
        {
            "Model": "LightGBM",
            "Objective": "binary",
            "random_state": random_state,
            "resampling": "none",
            "class_weighting": "none",
            "split_proportions": "stratified_train: 80%, validation: 10%, test: 10%",
            "preprocessing": "one_hot_categorical_passthrough_numeric_binary",
        }
    )
    
    lgbm_baseline = LGBMClassifier(random_state=random_state,objective='binary')
    lgbm_baseline.fit(X_train_processed, y_train)

    y_val_pred = lgbm_baseline.predict(X_val_processed)
    y_val_proba = lgbm_baseline.predict_proba(X_val_processed)[:, 1]
    
    validation_metrics = metric_logger(
        y_true=y_val,
        y_pred=y_val_pred,
        y_proba=y_val_proba,
        k=10
    )
    print("Validation metrics:")
    print(validation_metrics)

    mlflow.lightgbm.log_model(
        lgbm_baseline,
        artifact_path="model"
    )

    mlflow.sklearn.log_model(
        preprocessor,
        artifact_path="preprocessor"
    )

    print(f"MLflow run ID: {run.info.run_id}")

MLflow run started with ID: 216d90081993417587e3ac86586bcb95
[LightGBM] [Info] Number of positive: 37368, number of negative: 267519
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 870
[LightGBM] [Info] Number of data points in the train set: 304887, number of used features: 155
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.122563 -> initscore=-1.968376
[LightGBM] [Info] Start training from score -1.968376


2026/08/23 14:23:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Validation metrics:
{'accuracy': 0.8771220907349584, 'precision': 0.3125, 'recall': 0.002140869192892314, 'f1_score': 0.00425260472039124, 'roc_auc': 0.8562721289305181, 'precision_at_k': np.float64(0.39149829441091577), 'recall_at_k': np.float64(0.3194176835795333), 'lift_at_k': np.float64(3.19426065045909), 'pr_auc': 0.36492881778490055}


2026/08/23 14:23:11 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/23 14:23:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/23 14:23:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/23 14:23:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For 

MLflow run ID: 216d90081993417587e3ac86586bcb95
🏃 View run run0_lgbm_no_resampling_baseline at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/216d90081993417587e3ac86586bcb95
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


### 6.5 Top-K selection evaluation

The model is intended to rank customers for campaign outreach. Therefore, evaluate the quality of the highest-scored 1%, 5%, 10%, and 20% of validation customers. These percentages represent possible campaign capacities; they are evaluation scenarios, not fixed company requirements.

In [34]:
top_k_percentages = [1, 5, 10, 20]

top_k_results = []
for top_k in top_k_percentages:
    top_k_results.append({
        "selection_percentage": top_k,
        "customers_selected": int(len(y_val) * top_k / 100),
        "precision_at_k": precision_at_k(y_val, y_val_proba, top_k),
        "recall_at_k": recall_at_k(y_val, y_val_proba, top_k),
        "lift_at_k": Lift_at_k(y_val, y_val_proba, top_k),
    })

top_k_results = pd.DataFrame(top_k_results)
display(top_k_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
}))

,selection_percentage,customers_selected,precision_at_k,recall_at_k,lift_at_k
0,1,381,46.1942%,3.7679%,3.769
1,5,1905,41.6273%,16.9771%,3.396
2,10,3811,39.1498%,31.9418%,3.194
3,20,7622,35.2401%,57.5037%,2.875


### 6.6 Simple targeting baselines

These baselines provide reference points. Predict-none represents contacting nobody, while predict-all represents treating every customer as a positive response. Their performance prevents the LightGBM result from being interpreted without context.

In [36]:
baseline_predictions = {
    "predict_none": np.zeros(len(y_val), dtype=int),
    "predict_all": np.ones(len(y_val), dtype=int),
}

baseline_probabilities = {
    "predict_none": np.zeros(len(y_val), dtype=float),
    "predict_all": np.ones(len(y_val), dtype=float),
}

baseline_rows = []
for baseline_name in baseline_predictions:
    baseline_y_pred = baseline_predictions[baseline_name]
    baseline_y_proba = baseline_probabilities[baseline_name]
    baseline_rows.append({
        "model": baseline_name,
        "accuracy": accuracy_score(y_val, baseline_y_pred),
        "precision": precision_score(y_val, baseline_y_pred, zero_division=0),
        "recall": recall_score(y_val, baseline_y_pred, zero_division=0),
        "f1_score": f1_score(y_val, baseline_y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, baseline_y_proba),
        "pr_auc": pr_auc(y_val, baseline_y_proba),
        "precision_at_10pct": precision_at_k(y_val, baseline_y_proba, 10),
        "recall_at_10pct": recall_at_k(y_val, baseline_y_proba, 10),
        "lift_at_10pct": Lift_at_k(y_val, baseline_y_proba, 10),
    })

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results.style.format({
    "accuracy": "{:.4%}",
    "precision": "{:.4%}",
    "recall": "{:.4%}",
    "f1_score": "{:.4%}",
    "roc_auc": "{:.4f}",
    "pr_auc": "{:.4f}",
    "precision_at_10pct": "{:.4%}",
    "recall_at_10pct": "{:.4%}",
    "lift_at_10pct": "{:.3f}",
}))

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,precision_at_10pct,recall_at_10pct,lift_at_10pct
0,predict_none,87.7437%,0.0000%,0.0000%,0.0000%,0.5000,0.1226,11.9654%,9.7624%,0.976
1,predict_all,12.2563%,12.2563%,100.0000%,21.8363%,0.5000,0.1226,11.9654%,9.7624%,0.976


### 6.7 Ranking baseline comparison

This section compares LightGBM with an interpretable Logistic Regression model and a random-ranking baseline. Every candidate uses the same validation split and the same top-K metric definitions.

In [37]:
logistic_scaler = StandardScaler(with_mean=False)
X_train_logistic = logistic_scaler.fit_transform(X_train_processed)
X_val_logistic = logistic_scaler.transform(X_val_processed)

logistic_baseline = LogisticRegression(random_state=random_state, max_iter=1000)
logistic_baseline.fit(X_train_logistic, y_train)
y_logistic_proba = logistic_baseline.predict_proba(X_val_logistic)[:, 1]
y_logistic_pred = logistic_baseline.predict(X_val_logistic)

random_generator = np.random.default_rng(random_state)
y_random_proba = random_generator.random(len(y_val))
y_random_pred = (y_random_proba >= 0.5).astype(int)

model_scores = {
    "LightGBM": y_val_proba,
    "Logistic Regression": y_logistic_proba,
    "Random ranking": y_random_proba,
    "Predict none": np.zeros(len(y_val), dtype=float),
    "Predict all": np.ones(len(y_val), dtype=float),
}
model_predictions = {
    "LightGBM": y_val_pred,
    "Logistic Regression": y_logistic_pred,
    "Random ranking": y_random_pred,
    "Predict none": np.zeros(len(y_val), dtype=int),
    "Predict all": np.ones(len(y_val), dtype=int),
}

comparison_rows = []
for model_name, model_proba in model_scores.items():
    model_pred = model_predictions[model_name]
    for top_k in top_k_percentages:
        comparison_rows.append({
            "model": model_name,
            "selection_percentage": top_k,
            "accuracy": accuracy_score(y_val, model_pred),
            "precision": precision_score(y_val, model_pred, zero_division=0),
            "recall": recall_score(y_val, model_pred, zero_division=0),
            "f1_score": f1_score(y_val, model_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_val, model_proba),
            "pr_auc": pr_auc(y_val, model_proba),
            "precision_at_k": precision_at_k(y_val, model_proba, top_k),
            "recall_at_k": recall_at_k(y_val, model_proba, top_k),
            "lift_at_k": Lift_at_k(y_val, model_proba, top_k),
        })

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.style.format({
    "accuracy": "{:.4%}", "precision": "{:.4%}",
    "recall": "{:.4%}", "f1_score": "{:.4%}",
    "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
    "precision_at_k": "{:.4%}", "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
}))

,model,selection_percentage,accuracy,precision,recall,f1_score,roc_auc,pr_auc,precision_at_k,recall_at_k,lift_at_k
0,LightGBM,1,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,46.1942%,3.7679%,3.769
1,LightGBM,5,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,41.6273%,16.9771%,3.396
2,LightGBM,10,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,39.1498%,31.9418%,3.194
3,LightGBM,20,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,35.2401%,57.5037%,2.875
4,Logistic Regression,1,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,35.6955%,2.9116%,2.912
5,Logistic Regression,5,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,36.8504%,15.0289%,3.007
6,Logistic Regression,10,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,36.9194%,30.1220%,3.012
7,Logistic Regression,20,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,34.2430%,55.8767%,2.794
8,Random ranking,1,49.4582%,11.9372%,48.9831%,19.1962%,0.4883,0.1183,11.2861%,0.9206%,0.921
9,Random ranking,5,49.4582%,11.9372%,48.9831%,19.1962%,0.4883,0.1183,11.3911%,4.6457%,0.929


### 6.8 LightGBM threshold analysis

The default threshold of 0.5 is not assumed to be correct. This analysis shows how LightGBM precision, recall, F1, and contact rate change across candidate thresholds. It is descriptive only; do not freeze a threshold until the business operating objective is selected.

In [38]:
threshold_rows = []
for threshold in np.arange(0.05, 1.00, 0.05):
    threshold_pred = (y_val_proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "contact_rate": threshold_pred.mean(),
        "precision": precision_score(y_val, threshold_pred, zero_division=0),
        "recall": recall_score(y_val, threshold_pred, zero_division=0),
        "f1_score": f1_score(y_val, threshold_pred, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.style.format({
    "threshold": "{:.2f}", "contact_rate": "{:.4%}",
    "precision": "{:.4%}", "recall": "{:.4%}",
    "f1_score": "{:.4%}",
}))

,threshold,contact_rate,precision,recall,f1_score
0,0.05,47.7736%,25.1936%,98.2017%,40.0997%
1,0.10,43.0716%,27.0423%,95.0332%,42.1038%
2,0.15,37.9680%,29.0947%,90.1306%,43.9893%
3,0.20,31.7284%,31.6159%,81.8454%,45.6124%
4,0.25,25.9663%,33.3872%,70.7343%,45.3628%
5,0.30,17.7613%,36.1058%,52.3228%,42.7273%
6,0.35,9.6586%,39.3100%,30.9784%,34.6504%
7,0.40,3.3193%,42.8458%,11.6035%,18.2615%
8,0.45,0.6927%,46.5909%,2.6333%,4.9848%
9,0.50,0.0840%,31.2500%,0.2141%,0.4253%


currently the performance of lightgbm is the best among the candidates . We will now try improving using imbalanced-learn resampling techniques and hyperparameter tuning. 
One more important point ot mention that the threshold 0.5 is not the best threshold for the model , instead 0.35 works fine but this needs more analysis and validation.

APPLY CLASS WEIGHTING

In [39]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scaling_factor = neg_count / pos_count
print(f"Negative class count: {neg_count}, Positive class count: {pos_count}, Scaling factor: {scaling_factor:.4f}")

Negative class count: 267519, Positive class count: 37368, Scaling factor: 7.1590


In [40]:
run_name = "run1_lgbm_class_weighting"
with mlflow.start_run(run_name=run_name) as run:
    print(f"MLflow run started with ID: {run.info.run_id}")
    mlflow.log_params(
        {
            "Model": "LightGBM",
            "Objective": "binary",
            "random_state": random_state,
            "resampling": "none",
            "class_weighting": "scale_pos_weight",
            "scale_pos_weight": float(scaling_factor),
            "split_proportions": "stratified_train: 80%, validation: 10%, test: 10%",
            "preprocessing": "one_hot_categorical_passthrough_numeric_binary",
        }
    )
    
    lgbm_class_weighted = LGBMClassifier(random_state=random_state,objective='binary', scale_pos_weight=scaling_factor)
    lgbm_class_weighted.fit(X_train_processed, y_train)

    y_val_pred_weighted = lgbm_class_weighted.predict(X_val_processed)
    y_val_proba_weighted = lgbm_class_weighted.predict_proba(X_val_processed)[:, 1]
    
    weighted_validation_metrics = metric_logger(
        y_true=y_val,
        y_pred=y_val_pred_weighted,
        y_proba=y_val_proba_weighted,
        k=10
    )
    print("Weighted validation metrics:")
    print(weighted_validation_metrics)

    mlflow.lightgbm.log_model(
        lgbm_class_weighted,
        artifact_path="model"
    )

    mlflow.sklearn.log_model(
        preprocessor,
        artifact_path="preprocessor"
    )

    print(f"MLflow run ID: {run.info.run_id}")

MLflow run started with ID: d506a9f319484e8b867b5a90ade61515
[LightGBM] [Info] Number of positive: 37368, number of negative: 267519
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020399 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 870
[LightGBM] [Info] Number of data points in the train set: 304887, number of used features: 155
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.122563 -> initscore=-1.968376
[LightGBM] [Info] Start training from score -1.968376


2026/08/23 14:24:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Weighted validation metrics:
{'accuracy': 0.700952480911023, 'precision': 0.2816800830952999, 'recall': 0.9289231427959752, 'f1_score': 0.43227895392278953, 'roc_auc': 0.8564646310995565, 'precision_at_k': np.float64(0.39018630280766203), 'recall_at_k': np.float64(0.31834724898308714), 'lift_at_k': np.float64(3.183556023614388), 'pr_auc': 0.36839641116343086}


2026/08/23 14:24:32 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/23 14:24:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/23 14:24:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/23 14:24:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For 

MLflow run ID: d506a9f319484e8b867b5a90ade61515
🏃 View run run1_lgbm_class_weighting at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/d506a9f319484e8b867b5a90ade61515
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


### 6.9 Class-weighted top-K evaluation

Evaluate the class-weighted model at the same campaign capacities as the unweighted baseline. The original baseline probabilities remain stored separately.

In [41]:
baseline_proba_for_comparison = model_scores["LightGBM"]

weighted_comparison_rows = []
for model_name, model_proba in {
    "LightGBM baseline": baseline_proba_for_comparison,
    "LightGBM class weighted": y_val_proba_weighted,
}.items():
    for top_k in top_k_percentages:
        weighted_comparison_rows.append({
            "model": model_name,
            "selection_percentage": top_k,
            "precision_at_k": precision_at_k(y_val, model_proba, top_k),
            "recall_at_k": recall_at_k(y_val, model_proba, top_k),
            "lift_at_k": Lift_at_k(y_val, model_proba, top_k),
            "pr_auc": pr_auc(y_val, model_proba),
            "roc_auc": roc_auc_score(y_val, model_proba),
        })

weighted_comparison_results = pd.DataFrame(weighted_comparison_rows)
display(weighted_comparison_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))

,model,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,LightGBM baseline,1,46.1942%,3.7679%,3.769,0.3649,0.8563
1,LightGBM baseline,5,41.6273%,16.9771%,3.396,0.3649,0.8563
2,LightGBM baseline,10,39.1498%,31.9418%,3.194,0.3649,0.8563
3,LightGBM baseline,20,35.2401%,57.5037%,2.875,0.3649,0.8563
4,LightGBM class weighted,1,48.0315%,3.9178%,3.919,0.3684,0.8565
5,LightGBM class weighted,5,41.1549%,16.7844%,3.358,0.3684,0.8565
6,LightGBM class weighted,10,39.0186%,31.8347%,3.184,0.3684,0.8565
7,LightGBM class weighted,20,35.4369%,57.8249%,2.891,0.3684,0.8565


### 6.10 Class-weighted threshold analysis

Class weighting can change the score distribution, so the unweighted threshold must not be reused automatically.

In [42]:
weighted_threshold_rows = []
for threshold in np.arange(0.05, 1.00, 0.05):
    weighted_threshold_pred = (y_val_proba_weighted >= threshold).astype(int)
    weighted_threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "contact_rate": weighted_threshold_pred.mean(),
        "precision": precision_score(y_val, weighted_threshold_pred, zero_division=0),
        "recall": recall_score(y_val, weighted_threshold_pred, zero_division=0),
        "f1_score": f1_score(y_val, weighted_threshold_pred, zero_division=0),
    })

weighted_threshold_results = pd.DataFrame(weighted_threshold_rows)
display(weighted_threshold_results.style.format({
    "threshold": "{:.2f}",
    "contact_rate": "{:.4%}",
    "precision": "{:.4%}",
    "recall": "{:.4%}",
    "f1_score": "{:.4%}",
}))

,threshold,contact_rate,precision,recall,f1_score
0,0.05,54.6693%,22.3470%,99.6789%,36.5091%
1,0.10,53.5908%,22.7722%,99.5718%,37.0671%
2,0.15,52.1188%,23.3802%,99.4220%,37.8577%
3,0.20,50.0643%,24.2138%,98.9082%,38.9036%
4,0.25,48.1147%,25.0368%,98.2873%,39.9079%
5,0.30,46.9838%,25.5166%,97.8163%,40.4748%
6,0.35,45.9001%,25.9590%,97.2169%,40.9764%
7,0.40,44.4675%,26.5416%,96.2963%,41.6135%
8,0.45,42.4234%,27.2823%,94.4337%,42.3341%
9,0.50,40.4188%,28.1680%,92.8923%,43.2279%


### 6.11 LightGBM hyperparameter tuning with cross-validation

This experiment compares unweighted and class-weighted LightGBM using the same randomized hyperparameter search. Cross-validation is performed only inside the training split. The validation split is used only for final candidate comparison, and the final test split remains untouched.

In [43]:
from sklearn.model_selection import StratifiedKFold

cv_splitter = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=random_state,
)

lgbm_param_distributions = {
    "n_estimators": [200, 400, 700],
    "learning_rate": [0.03, 0.05, 0.10],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 5, 8, 12],
    "min_child_samples": [20, 50, 100],
    "subsample": [0.8, 1.0],
    "subsample_freq": [1],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0],
}

lgbm_variants = {
    "unweighted": None,
    "class_weighted": float(scaling_factor),
}

tuned_models = {}
tuned_validation_rows = []

with mlflow.start_run(run_name="tier1_lgbm_cv_comparison") as parent_run:
    mlflow.log_params({
        "experiment_stage": "tier1_lgbm_hyperparameter_tuning",
        "cv_strategy": "stratified_3_fold_on_training_only",
        "search_scoring": "average_precision",
        "n_iter": 12,
        "random_state": random_state,
        "validation_split": "untouched_during_cv",
        "test_split": "untouched",
    })

    for variant_name, variant_scale_pos_weight in lgbm_variants.items():
        with mlflow.start_run(
            run_name=f"tier1_lgbm_{variant_name}",
            nested=True,
        ) as child_run:
            estimator_params = {
                "objective": "binary",
                "random_state": random_state,
                "n_jobs": -1,
                "verbosity": -1,
            }
            if variant_scale_pos_weight is not None:
                estimator_params["scale_pos_weight"] = variant_scale_pos_weight

            search = RandomizedSearchCV(
                estimator=LGBMClassifier(**estimator_params),
                param_distributions=lgbm_param_distributions,
                n_iter=12,
                scoring="average_precision",
                cv=cv_splitter,
                refit=True,
                random_state=random_state,
                n_jobs=-1,
                verbose=0,
                return_train_score=False,
            )
            search.fit(X_train_processed, y_train)

            best_model = search.best_estimator_
            tuned_models[variant_name] = best_model
            tuned_proba = best_model.predict_proba(X_val_processed)[:, 1]
            tuned_pred = best_model.predict(X_val_processed)

            mlflow.log_params({
                "variant": variant_name,
                "class_weighting": "scale_pos_weight" if variant_scale_pos_weight is not None else "none",
                "scale_pos_weight": str(variant_scale_pos_weight),
            })
            mlflow.log_params({
                f"best_{key}": str(value)
                for key, value in search.best_params_.items()
            })
            mlflow.log_metric("best_cv_average_precision", float(search.best_score_))

            mlflow.log_metrics({
                "validation_accuracy_at_0_5": float(accuracy_score(y_val, tuned_pred)),
                "validation_precision_at_0_5": float(precision_score(y_val, tuned_pred, zero_division=0)),
                "validation_recall_at_0_5": float(recall_score(y_val, tuned_pred, zero_division=0)),
                "validation_f1_at_0_5": float(f1_score(y_val, tuned_pred, zero_division=0)),
                "validation_pr_auc": float(pr_auc(y_val, tuned_proba)),
                "validation_roc_auc": float(roc_auc_score(y_val, tuned_proba)),
            })

            for top_k in top_k_percentages:
                precision_value = float(precision_at_k(y_val, tuned_proba, top_k))
                recall_value = float(recall_at_k(y_val, tuned_proba, top_k))
                lift_value = float(Lift_at_k(y_val, tuned_proba, top_k))
                mlflow.log_metrics({
                    f"validation_precision_at_{top_k}pct": precision_value,
                    f"validation_recall_at_{top_k}pct": recall_value,
                    f"validation_lift_at_{top_k}pct": lift_value,
                })
                tuned_validation_rows.append({
                    "variant": variant_name,
                    "selection_percentage": top_k,
                    "precision_at_k": precision_value,
                    "recall_at_k": recall_value,
                    "lift_at_k": lift_value,
                    "pr_auc": float(pr_auc(y_val, tuned_proba)),
                    "roc_auc": float(roc_auc_score(y_val, tuned_proba)),
                })

            cv_results_for_logging = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
            mlflow.log_text(
                cv_results_for_logging.to_csv(index=False),
                artifact_file="cv_results.csv",
            )
            mlflow.lightgbm.log_model(best_model, artifact_path="model")
            print(f"Completed {variant_name}; MLflow run ID: {child_run.info.run_id}")

    tuned_validation_results = pd.DataFrame(tuned_validation_rows)
    mlflow.log_text(
        tuned_validation_results.to_csv(index=False),
        artifact_file="tuned_validation_results.csv",
    )

display(tuned_validation_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))

2026/08/23 14:28:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/23 14:28:40 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/23 14:28:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Completed unweighted; MLflow run ID: 8d990be7ad084ef4a6a83805055caac2
🏃 View run tier1_lgbm_unweighted at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/8d990be7ad084ef4a6a83805055caac2
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


2026/08/23 14:31:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/23 14:32:05 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/23 14:32:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Completed class_weighted; MLflow run ID: e573ee5423434e6485404a96ca5f5731
🏃 View run tier1_lgbm_class_weighted at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/e573ee5423434e6485404a96ca5f5731
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2
🏃 View run tier1_lgbm_cv_comparison at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/74d6fb1bd22049b684b8edcec25147a7
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


,variant,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,unweighted,1,50.3937%,4.1105%,4.112,0.3689,0.8571
1,unweighted,5,41.4173%,16.8915%,3.379,0.3689,0.8571
2,unweighted,10,39.2548%,32.0274%,3.203,0.3689,0.8571
3,unweighted,20,35.5681%,58.0390%,2.902,0.3689,0.8571
4,class_weighted,1,50.6562%,4.1319%,4.133,0.3700,0.8573
5,class_weighted,5,41.4698%,16.9129%,3.384,0.3700,0.8573
6,class_weighted,10,39.6484%,32.3485%,3.235,0.3700,0.8573
7,class_weighted,20,35.5025%,57.9319%,2.897,0.3700,0.8573


LGBM , its done bro!!

Time to test out catboost

## 7. CATBOOST

### 7.1 Catboost data prep


In [42]:
from catboost import CatBoostClassifier

In [43]:
catboost_categorical_features = [
    "Gender",
    "Vehicle_Age",
    "Vehicle_Damage",
    "Region_Code",
    "Policy_Sales_Channel",
]

In [44]:
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()
X_test_cat = X_test.copy()

In [45]:
for feature in catboost_categorical_features:
    X_train_cat[feature] = X_train_cat[feature].astype(str)
    X_test_cat[feature] = X_test_cat[feature].astype(str)
    X_val_cat[feature] = X_val_cat[feature].astype(str)

### 7.2 Catboost model training

In [46]:
catboost_baseline = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=7,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=random_state,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1
)

In [47]:
with mlflow.start_run(run_name="tier1_catboost_unweighted") as run:
    mlflow.log_params({
        "model": "CatBoost",
        "variant": "unweighted",
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 7,
        "random_seed": random_state,
        "class_weighting": "none",
        "categorical_features": ",".join(catboost_categorical_features),
        "preprocessing": "native_catboost_categorical_features",
        "split_proportions": "stratified_train: 80%, validation: 10%, test: 10%",
    })

🏃 View run tier1_catboost_unweighted at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/77d540babcc646c4b7541b648ac807dd
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


In [48]:
catboost_baseline.fit(
        X_train_cat,
        y_train,
        cat_features=catboost_categorical_features,
        eval_set=(X_val_cat, y_val),
        use_best_model=True,
        early_stopping_rounds=50,
        verbose=False,
    )

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [49]:
y_val_proba_catboost = catboost_baseline.predict_proba(X_val_cat)[:, 1]
y_val_pred_catboost = (
        catboost_baseline.predict(X_val_cat)
        .ravel()
        .astype(int)
    )

In [50]:
catboost_validation_metrics = {
        "validation_accuracy_at_0_5": accuracy_score(
            y_val, y_val_pred_catboost
        ),
        "validation_precision_at_0_5": precision_score(
            y_val, y_val_pred_catboost, zero_division=0
        ),
        "validation_recall_at_0_5": recall_score(
            y_val, y_val_pred_catboost, zero_division=0
        ),
        "validation_f1_at_0_5": f1_score(
            y_val, y_val_pred_catboost, zero_division=0
        ),
        "validation_roc_auc": roc_auc_score(
            y_val, y_val_proba_catboost
        ),
        "validation_pr_auc": pr_auc(
            y_val, y_val_proba_catboost
        ),
    }

In [51]:
mlflow.log_metrics(catboost_validation_metrics)

In [52]:
top_k_percentages = [1, 5, 10, 20]

In [53]:
for top_k in top_k_percentages:
        mlflow.log_metrics({
            f"validation_precision_at_{top_k}pct": float(
                precision_at_k(y_val, y_val_proba_catboost, top_k)
            ),
            f"validation_recall_at_{top_k}pct": float(
                recall_at_k(y_val, y_val_proba_catboost, top_k)
            ),
            f"validation_lift_at_{top_k}pct": float(
                Lift_at_k(y_val, y_val_proba_catboost, top_k)
            ),
        })

In [54]:
mlflow.catboost.log_model(
        catboost_baseline,
        name="model",
        input_example=X_val_cat.head(5),
    )

print("CatBoost validation metrics:")
print(catboost_validation_metrics)
print(f"MLflow run ID: {run.info.run_id}")

2026/08/23 04:06:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


CatBoost validation metrics:
{'validation_accuracy_at_0_5': 0.8773582430269476, 'validation_precision_at_0_5': 0.4881889763779528, 'validation_recall_at_0_5': 0.01327338899593235, 'validation_f1_at_0_5': 0.025844101709045435, 'validation_roc_auc': 0.8576094007205203, 'validation_pr_auc': 0.3698637130107868}
MLflow run ID: 77d540babcc646c4b7541b648ac807dd


### 7.3 Catboost top-K evaluation

In [55]:
catboost_top_k_results = []

for top_k in top_k_percentages:
    catboost_top_k_results.append({
        "model": "CatBoost unweighted",
        "selection_percentage": top_k,
        "precision_at_k": precision_at_k(
            y_val, y_val_proba_catboost, top_k
        ),
        "recall_at_k": recall_at_k(
            y_val, y_val_proba_catboost, top_k
        ),
        "lift_at_k": Lift_at_k(
            y_val, y_val_proba_catboost, top_k
        ),
        "pr_auc": pr_auc(
            y_val, y_val_proba_catboost
        ),
        "roc_auc": roc_auc_score(
            y_val, y_val_proba_catboost
        ),
    })

catboost_top_k_results = pd.DataFrame(catboost_top_k_results)

display(
    catboost_top_k_results.style.format({
        "precision_at_k": "{:.4%}",
        "recall_at_k": "{:.4%}",
        "lift_at_k": "{:.3f}",
        "pr_auc": "{:.4f}",
        "roc_auc": "{:.4f}",
    })
)

,model,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,CatBoost unweighted,1,50.9186%,4.1533%,4.154,0.3699,0.8576
1,CatBoost unweighted,5,42.4672%,17.3196%,3.465,0.3699,0.8576
2,CatBoost unweighted,10,39.5959%,32.3057%,3.231,0.3699,0.8576
3,CatBoost unweighted,20,35.4631%,57.8677%,2.893,0.3699,0.8576


### 7.4 CatBoost with balanced class weights

This is a separate CatBoost variant. It uses the same native categorical-feature setup and core hyperparameters as the unweighted model, with `auto_class_weights="Balanced"` added to increase the training influence of positive responses. The validation set remains the decision set; the test set is not used here.

In [56]:
catboost_balanced = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=random_state,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)

In [57]:
if mlflow.active_run() is not None:
    stale_run_id = mlflow.active_run().info.run_id
    print(f"Closing stale active MLflow run: {stale_run_id}")
    try:
        mlflow.end_run(status="KILLED")
    except Exception as cleanup_error:
        print(f"Stale run was already unavailable remotely; local context cleared: {cleanup_error}")

with mlflow.start_run(run_name="tier1_catboost_balanced") as balanced_run:
    mlflow.log_params({
        "model": "CatBoost",
        "variant": "balanced",
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 7,
        "random_seed": random_state,
        "class_weighting": "auto_class_weights=Balanced",
        "categorical_features": ",".join(catboost_categorical_features),
        "preprocessing": "native_catboost_categorical_features",
        "split_proportions": "stratified_train: 80%, validation: 10%, test: 10%",
    })

    catboost_balanced.fit(
        X_train_cat,
        y_train,
        cat_features=catboost_categorical_features,
        eval_set=(X_val_cat, y_val),
        use_best_model=True,
        early_stopping_rounds=50,
        verbose=False,
    )

    y_val_proba_catboost_balanced = catboost_balanced.predict_proba(X_val_cat)[:, 1]
    y_val_pred_catboost_balanced = (
        catboost_balanced.predict(X_val_cat).ravel().astype(int)
    )

    catboost_balanced_validation_metrics = {
        "validation_accuracy_at_0_5": float(accuracy_score(y_val, y_val_pred_catboost_balanced)),
        "validation_precision_at_0_5": float(precision_score(y_val, y_val_pred_catboost_balanced, zero_division=0)),
        "validation_recall_at_0_5": float(recall_score(y_val, y_val_pred_catboost_balanced, zero_division=0)),
        "validation_f1_at_0_5": float(f1_score(y_val, y_val_pred_catboost_balanced, zero_division=0)),
        "validation_roc_auc": float(roc_auc_score(y_val, y_val_proba_catboost_balanced)),
        "validation_pr_auc": float(pr_auc(y_val, y_val_proba_catboost_balanced)),
    }
    mlflow.log_metrics(catboost_balanced_validation_metrics)

    for top_k in top_k_percentages:
        mlflow.log_metrics({
            f"validation_precision_at_{top_k}pct": float(precision_at_k(y_val, y_val_proba_catboost_balanced, top_k)),
            f"validation_recall_at_{top_k}pct": float(recall_at_k(y_val, y_val_proba_catboost_balanced, top_k)),
            f"validation_lift_at_{top_k}pct": float(Lift_at_k(y_val, y_val_proba_catboost_balanced, top_k)),
        })

    mlflow.catboost.log_model(
        catboost_balanced,
        name="model",
        input_example=X_val_cat.head(5),
    )

print("Balanced CatBoost validation metrics:")
print(catboost_balanced_validation_metrics)
print(f"MLflow run ID: {balanced_run.info.run_id}")

Closing stale active MLflow run: 3b92f914e64044dbbce9551aaa6baef5
🏃 View run shivering-rook-213 at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/3b92f914e64044dbbce9551aaa6baef5
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


2026/08/23 04:09:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run tier1_catboost_balanced at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/1bd77b92e644497fba869489f4eb2715
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2
Balanced CatBoost validation metrics:
{'validation_accuracy_at_0_5': 0.705019548161948, 'validation_precision_at_0_5': 0.2839197632357777, 'validation_recall_at_0_5': 0.9242132305716121, 'validation_f1_at_0_5': 0.43439323807607166, 'validation_roc_auc': 0.857501454561844, 'validation_pr_auc': 0.36965175853733484}
MLflow run ID: 1bd77b92e644497fba869489f4eb2715


### 7.5 CatBoost versus tuned LightGBM

Compare the two CatBoost variants with the already tuned class-weighted LightGBM on the same validation split. The comparison uses ranking metrics because the project selects the highest-scoring customers for outreach.

In [58]:
comparison_probas = {
    "LightGBM tuned weighted": tuned_models["class_weighted"].predict_proba(X_val_processed)[:, 1],
    "CatBoost unweighted": y_val_proba_catboost,
    "CatBoost balanced": y_val_proba_catboost_balanced,
}

comparison_rows = []
for model_name, model_proba in comparison_probas.items():
    for top_k in top_k_percentages:
        comparison_rows.append({
            "model": model_name,
            "selection_percentage": top_k,
            "precision_at_k": float(precision_at_k(y_val, model_proba, top_k)),
            "recall_at_k": float(recall_at_k(y_val, model_proba, top_k)),
            "lift_at_k": float(Lift_at_k(y_val, model_proba, top_k)),
            "pr_auc": float(pr_auc(y_val, model_proba)),
            "roc_auc": float(roc_auc_score(y_val, model_proba)),
        })

catboost_lgbm_comparison = pd.DataFrame(comparison_rows)

if mlflow.active_run() is not None:
    stale_run_id = mlflow.active_run().info.run_id
    print(f"Closing stale active MLflow run: {stale_run_id}")
    try:
        mlflow.end_run(status="KILLED")
    except Exception as cleanup_error:
        print(f"Stale run was already unavailable remotely; local context cleared: {cleanup_error}")

with mlflow.start_run(run_name="tier1_catboost_lgbm_comparison") as comparison_run:
    mlflow.log_params({
        "comparison_split": "validation",
        "models": ",".join(comparison_probas.keys()),
        "selection_percentages": ",".join(map(str, top_k_percentages)),
        "test_set_used": False,
    })

    for row in catboost_lgbm_comparison.itertuples(index=False):
        model_key = row.model.lower().replace(" ", "_")
        mlflow.log_metrics({
            f"{model_key}_pr_auc": row.pr_auc,
            f"{model_key}_roc_auc": row.roc_auc,
            f"{model_key}_precision_at_{int(row.selection_percentage)}pct": row.precision_at_k,
            f"{model_key}_recall_at_{int(row.selection_percentage)}pct": row.recall_at_k,
            f"{model_key}_lift_at_{int(row.selection_percentage)}pct": row.lift_at_k,
        })

    mlflow.log_text(
        catboost_lgbm_comparison.to_csv(index=False),
        artifact_file="catboost_lgbm_validation_comparison.csv",
    )

display(catboost_lgbm_comparison.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))
print(f"Comparison MLflow run ID: {comparison_run.info.run_id}")

🏃 View run tier1_catboost_lgbm_comparison at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/58bc0769553b43a3993bf27e4d3bcc4a
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


,model,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,LightGBM tuned weighted,1,50.6562%,4.1319%,4.133,0.3700,0.8573
1,LightGBM tuned weighted,5,41.4698%,16.9129%,3.384,0.3700,0.8573
2,LightGBM tuned weighted,10,39.6484%,32.3485%,3.235,0.3700,0.8573
3,LightGBM tuned weighted,20,35.5025%,57.9319%,2.897,0.3700,0.8573
4,CatBoost unweighted,1,50.9186%,4.1533%,4.154,0.3699,0.8576
5,CatBoost unweighted,5,42.4672%,17.3196%,3.465,0.3699,0.8576
6,CatBoost unweighted,10,39.5959%,32.3057%,3.231,0.3699,0.8576
7,CatBoost unweighted,20,35.4631%,57.8677%,2.893,0.3699,0.8576
8,CatBoost balanced,1,50.1312%,4.0891%,4.090,0.3697,0.8575
9,CatBoost balanced,5,41.7848%,17.0413%,3.409,0.3697,0.8575


Comparison MLflow run ID: 58bc0769553b43a3993bf27e4d3bcc4a


### 7.6 Cross-validated CatBoost tuning

This section applies the same training-only, stratified 3-fold CV protocol used for LightGBM. Average Precision is the search objective because the product is a minority-class ranking problem. Both unweighted and balanced variants are evaluated, and the final test set is not used.

In [59]:
catboost_cv_splitter = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=random_state,
)

catboost_param_distributions = {
    "iterations": [300, 500, 800],
    "learning_rate": [0.03, 0.05, 0.10],
    "depth": [5, 7, 9],
    "l2_leaf_reg": [3, 5, 10],
    "random_strength": [0, 1, 2],
    "border_count": [64, 128, 254],
}

catboost_cv_variants = {
    "unweighted": None,
    "balanced": "Balanced",
}

catboost_tuned_models = {}
catboost_tuned_validation_rows = []

if mlflow.active_run() is not None:
    stale_run_id = mlflow.active_run().info.run_id
    print(f"Closing stale active MLflow run: {stale_run_id}")
    try:
        mlflow.end_run(status="KILLED")
    except Exception as cleanup_error:
        print(f"Stale run was already unavailable remotely; local context cleared: {cleanup_error}")

with mlflow.start_run(run_name="tier1_catboost_cv_comparison") as catboost_parent_run:
    mlflow.log_params({
        "experiment_stage": "tier1_catboost_hyperparameter_tuning",
        "cv_strategy": "stratified_3_fold_on_training_only",
        "search_scoring": "average_precision",
        "n_iter": 12,
        "random_state": random_state,
        "validation_split": "untouched_during_cv",
        "test_split": "untouched",
        "categorical_features": ",".join(catboost_categorical_features),
    })

    for variant_name, auto_class_weight in catboost_cv_variants.items():
        estimator_params = {
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "random_seed": random_state,
            "verbose": False,
            "allow_writing_files": False,
            "thread_count": -1,
        }
        if auto_class_weight is not None:
            estimator_params["auto_class_weights"] = auto_class_weight

        with mlflow.start_run(
            run_name=f"tier1_catboost_cv_{variant_name}",
            nested=True,
        ) as catboost_child_run:
            search = RandomizedSearchCV(
                estimator=CatBoostClassifier(**estimator_params),
                param_distributions=catboost_param_distributions,
                n_iter=12,
                scoring="average_precision",
                cv=catboost_cv_splitter,
                refit=True,
                random_state=random_state,
                n_jobs=1,
                verbose=0,
                return_train_score=False,
            )
            search.fit(
                X_train_cat,
                y_train,
                cat_features=catboost_categorical_features,
            )

            best_model = search.best_estimator_
            catboost_tuned_models[variant_name] = best_model
            tuned_proba = best_model.predict_proba(X_val_cat)[:, 1]
            tuned_pred = best_model.predict(X_val_cat).ravel().astype(int)

            mlflow.log_params({
                "model": "CatBoost",
                "variant": variant_name,
                "class_weighting": auto_class_weight or "none",
                "categorical_features": ",".join(catboost_categorical_features),
            })
            mlflow.log_params({
                f"best_{key}": str(value)
                for key, value in search.best_params_.items()
            })
            mlflow.log_metrics({
                "best_cv_average_precision": float(search.best_score_),
                "validation_accuracy_at_0_5": float(accuracy_score(y_val, tuned_pred)),
                "validation_precision_at_0_5": float(precision_score(y_val, tuned_pred, zero_division=0)),
                "validation_recall_at_0_5": float(recall_score(y_val, tuned_pred, zero_division=0)),
                "validation_f1_at_0_5": float(f1_score(y_val, tuned_pred, zero_division=0)),
                "validation_pr_auc": float(pr_auc(y_val, tuned_proba)),
                "validation_roc_auc": float(roc_auc_score(y_val, tuned_proba)),
            })

            for top_k in top_k_percentages:
                precision_value = float(precision_at_k(y_val, tuned_proba, top_k))
                recall_value = float(recall_at_k(y_val, tuned_proba, top_k))
                lift_value = float(Lift_at_k(y_val, tuned_proba, top_k))
                mlflow.log_metrics({
                    f"validation_precision_at_{top_k}pct": precision_value,
                    f"validation_recall_at_{top_k}pct": recall_value,
                    f"validation_lift_at_{top_k}pct": lift_value,
                })
                catboost_tuned_validation_rows.append({
                    "variant": variant_name,
                    "selection_percentage": top_k,
                    "precision_at_k": precision_value,
                    "recall_at_k": recall_value,
                    "lift_at_k": lift_value,
                    "pr_auc": float(pr_auc(y_val, tuned_proba)),
                    "roc_auc": float(roc_auc_score(y_val, tuned_proba)),
                })

            cv_results_for_logging = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
            mlflow.log_text(
                cv_results_for_logging.to_csv(index=False),
                artifact_file="cv_results.csv",
            )
            mlflow.catboost.log_model(
                best_model,
                name="model",
                input_example=X_val_cat.head(5),
            )
            print(
                f"Completed CatBoost {variant_name}; MLflow run ID: {catboost_child_run.info.run_id}"
            )

    catboost_tuned_validation_results = pd.DataFrame(catboost_tuned_validation_rows)
    mlflow.log_text(
        catboost_tuned_validation_results.to_csv(index=False),
        artifact_file="catboost_tuned_validation_results.csv",
    )

display(catboost_tuned_validation_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))

2026/08/23 05:17:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Completed CatBoost unweighted; MLflow run ID: c76d4938509a4701aa6092bc478f36f1
🏃 View run tier1_catboost_cv_unweighted at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/c76d4938509a4701aa6092bc478f36f1
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


2026/08/23 06:50:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Completed CatBoost balanced; MLflow run ID: 95b76a8a22b34c2e969f3896070bcab4
🏃 View run tier1_catboost_cv_balanced at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/95b76a8a22b34c2e969f3896070bcab4
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2
🏃 View run tier1_catboost_cv_comparison at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/25fc161b580b4573bfa46c8915465b04
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


,variant,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,unweighted,1,49.0814%,4.0034%,4.005,0.3685,0.8571
1,unweighted,5,42.2047%,17.2126%,3.444,0.3685,0.8571
2,unweighted,10,39.5172%,32.2415%,3.224,0.3685,0.8571
3,unweighted,20,35.3844%,57.7392%,2.887,0.3685,0.8571
4,balanced,1,50.9186%,4.1533%,4.154,0.3695,0.8573
5,balanced,5,41.3648%,16.8700%,3.375,0.3695,0.8573
6,balanced,10,39.5434%,32.2629%,3.226,0.3695,0.8573
7,balanced,20,35.3188%,57.6322%,2.882,0.3695,0.8573


### 7.7 Tuned CatBoost versus tuned LightGBM

This final validation table compares the cross-validated CatBoost variants with the already tuned weighted LightGBM using the same top-K selection metrics.

In [60]:
cv_comparison_probas = {
    "LightGBM tuned weighted": tuned_models["class_weighted"].predict_proba(X_val_processed)[:, 1],
    "CatBoost CV unweighted": catboost_tuned_models["unweighted"].predict_proba(X_val_cat)[:, 1],
    "CatBoost CV balanced": catboost_tuned_models["balanced"].predict_proba(X_val_cat)[:, 1],
}

cv_comparison_rows = []
for model_name, model_proba in cv_comparison_probas.items():
    for top_k in top_k_percentages:
        cv_comparison_rows.append({
            "model": model_name,
            "selection_percentage": top_k,
            "precision_at_k": float(precision_at_k(y_val, model_proba, top_k)),
            "recall_at_k": float(recall_at_k(y_val, model_proba, top_k)),
            "lift_at_k": float(Lift_at_k(y_val, model_proba, top_k)),
            "pr_auc": float(pr_auc(y_val, model_proba)),
            "roc_auc": float(roc_auc_score(y_val, model_proba)),
        })

catboost_cv_comparison = pd.DataFrame(cv_comparison_rows)

if mlflow.active_run() is not None:
    try:
        mlflow.end_run(status="KILLED")
    except Exception as cleanup_error:
        print(f"Stale run was already unavailable remotely; local context cleared: {cleanup_error}")

with mlflow.start_run(run_name="tier1_catboost_cv_lgbm_comparison") as cv_comparison_run:
    mlflow.log_params({
        "comparison_split": "validation",
        "models": ",".join(cv_comparison_probas.keys()),
        "selection_percentages": ",".join(map(str, top_k_percentages)),
        "test_set_used": False,
    })
    mlflow.log_text(
        catboost_cv_comparison.to_csv(index=False),
        artifact_file="catboost_cv_lgbm_validation_comparison.csv",
    )

display(catboost_cv_comparison.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))
print(f"CatBoost CV comparison MLflow run ID: {cv_comparison_run.info.run_id}")

🏃 View run tier1_catboost_cv_lgbm_comparison at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/73ac62c57be94e6093320d48a47ad05a
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


,model,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,LightGBM tuned weighted,1,50.6562%,4.1319%,4.133,0.3700,0.8573
1,LightGBM tuned weighted,5,41.4698%,16.9129%,3.384,0.3700,0.8573
2,LightGBM tuned weighted,10,39.6484%,32.3485%,3.235,0.3700,0.8573
3,LightGBM tuned weighted,20,35.5025%,57.9319%,2.897,0.3700,0.8573
4,CatBoost CV unweighted,1,49.0814%,4.0034%,4.005,0.3685,0.8571
5,CatBoost CV unweighted,5,42.2047%,17.2126%,3.444,0.3685,0.8571
6,CatBoost CV unweighted,10,39.5172%,32.2415%,3.224,0.3685,0.8571
7,CatBoost CV unweighted,20,35.3844%,57.7392%,2.887,0.3685,0.8571
8,CatBoost CV balanced,1,50.9186%,4.1533%,4.154,0.3695,0.8573
9,CatBoost CV balanced,5,41.3648%,16.8700%,3.375,0.3695,0.8573


CatBoost CV comparison MLflow run ID: 73ac62c57be94e6093320d48a47ad05a


## 8. Feature Ablation

Hey so its time to run feature ablation to see which features are important and which are not. This will help us in feature selection and also in understanding the model better.

### 8.1 Baseline for Comparison

In [44]:
baseline_pr_auc = pr_auc(
    y_val,
    tuned_models["class_weighted"].predict_proba(X_val_processed)[:, 1]
)
baseline_top10_lift = Lift_at_k(
    y_val,
    tuned_models["class_weighted"].predict_proba(X_val_processed)[:, 1],
    10
)

print("Baseline PR-AUC:", baseline_pr_auc)
print("Baseline Top-10% Lift:", baseline_top10_lift)

Baseline PR-AUC: 0.3700067420799737
Baseline Top-10% Lift: 3.234938232468958


### 8.2 Domain Driven Interactions

In [45]:
def add_domain_interactions(dataframe):
    """
    Add domain-specific interaction features to the DataFrame.
    This function creates new features based on existing ones that may capture important relationships.
    
    Parameters:
    - data frame: pd.DataFrame, the input DataFrame containing the original features.
    
    Returns:
    - pd.DataFrame, the DataFrame with added interaction features.
    """
    dataframe = dataframe.copy()
    dataframe["Previously_Insured_Damage"] =  (
        dataframe["Previously_Insured"].astype(str)
        + "_"
        + dataframe["Vehicle_Damage"].astype(str)
    )
    
    dataframe["Vehicle_Age_Damage"] = (
        dataframe["Vehicle_Age"].astype(str)
        + "_"
        + dataframe["Vehicle_Damage"].astype(str)
    )
    
    return dataframe    

In [46]:
X_train_interactions = add_domain_interactions(X_train)
X_val_interactions = add_domain_interactions(X_val)
X_test_interactions = add_domain_interactions(X_test)

In [47]:
interaction_categorical_features = categorical_features + [
    "Previously_Insured_Damage",
    "Vehicle_Age_Damage",
]

### 8.3 Reference Ablation Experiment

Train one fresh model for each feature configuration

In [48]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


def evaluate_feature_variant(
    variant_name,
    train_data,
    validation_data,
    dropped_features=None,
    added_categorical_features=None,
):
    dropped_features = dropped_features or []
    added_categorical_features = added_categorical_features or []

    train_data = train_data.drop(columns=dropped_features).copy()
    validation_data = validation_data.drop(columns=dropped_features).copy()

    current_categorical_features = [
        feature
        for feature in categorical_features
        if feature not in dropped_features
    ]

    current_numerical_features = [
        feature
        for feature in numerical_features
        if feature not in dropped_features
    ]

    current_binary_features = [
        feature
        for feature in binary_features
        if feature not in dropped_features
    ]

    current_categorical_features += added_categorical_features

    variant_preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                current_categorical_features,
            ),
            (
                "numerical",
                "passthrough",
                current_numerical_features,
            ),
            (
                "binary",
                "passthrough",
                current_binary_features,
            ),
        ]
    )

    X_train_variant = variant_preprocessor.fit_transform(train_data)
    X_val_variant = variant_preprocessor.transform(validation_data)

    variant_model = clone(tuned_models["class_weighted"])
    variant_model.fit(X_train_variant, y_train)

    validation_proba = variant_model.predict_proba(X_val_variant)[:, 1]

    results = {
        "variant": variant_name,
        "pr_auc": pr_auc(y_val, validation_proba),
        "roc_auc": roc_auc_score(y_val, validation_proba),
        "precision_at_10": precision_at_k(
            y_val,
            validation_proba,
            10,
        ),
        "recall_at_10": recall_at_k(
            y_val,
            validation_proba,
            10,
        ),
        "lift_at_10": Lift_at_k(
            y_val,
            validation_proba,
            10,
        ),
    }

    return results

Now lets run feature ablations

In [49]:
ablation_results = []

ablation_results.append(
    evaluate_feature_variant(
        variant_name="all_features",
        train_data=X_train,
        validation_data=X_val,
    )
)

ablation_results.append(
    evaluate_feature_variant(
        variant_name="without_region_code",
        train_data=X_train,
        validation_data=X_val,
        dropped_features=["Region_Code"],
    )
)

ablation_results.append(
    evaluate_feature_variant(
        variant_name="without_policy_sales_channel",
        train_data=X_train,
        validation_data=X_val,
        dropped_features=["Policy_Sales_Channel"],
    )
)

ablation_results.append(
    evaluate_feature_variant(
        variant_name="without_region_and_channel",
        train_data=X_train,
        validation_data=X_val,
        dropped_features=[
            "Region_Code",
            "Policy_Sales_Channel",
        ],
    )
)

X_train_with_interactions = add_domain_interactions(X_train)
X_val_with_interactions = add_domain_interactions(X_val)

ablation_results.append(
    evaluate_feature_variant(
        variant_name="with_domain_interactions",
        train_data=X_train_with_interactions,
        validation_data=X_val_with_interactions,
        added_categorical_features=[
            "Previously_Insured_Damage",
            "Vehicle_Age_Damage",
        ],
    )
)

ablation_results_df = pd.DataFrame(ablation_results)
ablation_results_df

,variant,pr_auc,roc_auc,precision_at_10,recall_at_10,lift_at_10
0,all_features,0.370007,0.857251,0.396484,0.323485,3.234938
1,without_region_code,0.361094,0.853345,0.383889,0.313209,3.132174
2,without_policy_sales_channel,0.348680,0.849580,0.367882,0.300150,3.001577
3,without_region_and_channel,0.333514,0.843137,0.348990,0.284736,2.847431
4,with_domain_interactions,0.370212,0.857337,0.395434,0.322629,3.226375


### 8.4 Comparison of Ablation Experiments against baseline

In [50]:
ablation_results_df["delta_pr_auc"] = (
    ablation_results_df["pr_auc"] - baseline_pr_auc
)

ablation_results_df["delta_top10_lift"] = (
    ablation_results_df["lift_at_10"] - baseline_top10_lift
)

ablation_results_df.sort_values(
    "delta_pr_auc",
    ascending=False,
)

,variant,pr_auc,roc_auc,precision_at_10,recall_at_10,lift_at_10,delta_pr_auc,delta_top10_lift
4,with_domain_interactions,0.370212,0.857337,0.395434,0.322629,3.226375,0.000205,-0.008564
0,all_features,0.370007,0.857251,0.396484,0.323485,3.234938,0.000000,0.000000
1,without_region_code,0.361094,0.853345,0.383889,0.313209,3.132174,-0.008913,-0.102764
2,without_policy_sales_channel,0.348680,0.849580,0.367882,0.300150,3.001577,-0.021327,-0.233361
3,without_region_and_channel,0.333514,0.843137,0.348990,0.284736,2.847431,-0.036493,-0.387507


### 8.5 Analysis of Ablation Experiments

The `all_features` experiment is the reference configuration. It achieved PR-AUC `0.370007`, ROC-AUC `0.857251`, Precision@10 `39.6484%`, Recall@10 `32.3485%`, and Lift@10 `3.234938`.

#### Findings

1. **`Policy_Sales_Channel` is highly informative.** Removing it reduced PR-AUC from `0.370007` to `0.348680`, ROC-AUC from `0.857251` to `0.849580`, and Lift@10 from `3.234938` to `3.001577`. The top-10% recall also declined from `32.3485%` to `30.0150%`. This feature should remain in the model.
2. **`Region_Code` contributes meaningful additional signal.** Removing it reduced PR-AUC to `0.361094` and Lift@10 to `3.132174`. Its impact is smaller than the sales channel, but it is not redundant.
3. **The two features provide complementary information.** Removing both produced the largest degradation: PR-AUC `0.333514`, ROC-AUC `0.843137`, Precision@10 `34.8990%`, Recall@10 `28.4736%`, and Lift@10 `2.847431`. The combined drop is much larger than either individual drop, so both features should be retained.
4. **The proposed interactions did not produce a meaningful campaign improvement.** Adding `Previously_Insured × Vehicle_Damage` and `Vehicle_Age × Vehicle_Damage` increased PR-AUC by only `0.000205`, while Precision@10 decreased from `39.6484%` to `39.5434%` and Lift@10 decreased from `3.234938` to `3.226375`. This is not a consistent improvement and is currently more consistent with validation noise than a useful production feature change.
5. **The current feature set should remain the reference configuration.** The ablation results do not justify dropping either region or channel, and they do not yet justify retaining the tested interactions.

#### What these experiments establish

These are predictive ablation results, not causal effects. They show how much validation performance changes when information is removed or added under the same model protocol. They do not mean that changing a customer’s region or sales channel would change the customer’s response.

The results also support the earlier conclusion that the model is not failing because of the LightGBM-versus-CatBoost choice. Region and sales-channel information carries substantial signal, and removing it hurts every primary metric. The current model’s remaining limitation is more likely missing customer behavior, campaign history, and response-intent information than an incorrect treatment of these two feature groups.

#### Further ablations

A focused second ablation pass is justified for the remaining raw features: `Gender`, `Vehicle_Age`, `Vehicle_Damage`, `Age`, `Annual_Premium`, `Vintage`, `Driving_License`, and `Previously_Insured`. Run one leave-one-feature-out experiment for each, using the same split, tuned weighted LightGBM configuration, PR-AUC, and top-K metrics. Do not generate a large combinatorial interaction grid. After that pass, freeze the feature set unless a change produces a consistent improvement in both PR-AUC and the baseline top-10% selection metric.

### 8.6 Permuatation Importance
Lets measure how much validation performance drops when a feature is randomly shuffled.

In [52]:
from sklearn.inspection import permutation_importance

best_lgbm = tuned_models["class_weighted"]

# ColumnTransformer returned a sparse matrix. Convert only the validation
# matrix used for permutation importance to a dense array.
X_val_for_permutation = (
    X_val_processed.toarray()
    if hasattr(X_val_processed, "toarray")
    else X_val_processed
)

permutation_result = permutation_importance(
    best_lgbm,
    X_val_for_permutation,
    y_val,
    scoring="average_precision",
    n_repeats=5,
    random_state=random_state,
    n_jobs=1,
)

permutation_importance_df = pd.DataFrame({
    "feature": preprocessor.get_feature_names_out(),
    "importance_mean": permutation_result.importances_mean,
    "importance_std": permutation_result.importances_std,
})

permutation_importance_df.sort_values(
    "importance_mean",
    ascending=False,
).head(25)

,feature,importance_mean,importance_std
218,binary__Previously_Insured,0.108468,0.004709
214,numerical__Age,0.061564,0.002071
5,categorical__Vehicle_Damage_No,0.053954,0.004983
204,categorical__Policy_Sales_Channel_152.0,0.031803,0.003067
3,categorical__Vehicle_Age_< 1 Year,0.020390,0.003703
84,categorical__Policy_Sales_Channel_26.0,0.014765,0.001482
6,categorical__Vehicle_Damage_Yes,0.013936,0.003862
212,categorical__Policy_Sales_Channel_160.0,0.008601,0.001088
2,categorical__Vehicle_Age_1-2 Year,0.008235,0.001146
215,numerical__Annual_Premium,0.007198,0.002195


In [53]:
def assign_feature_group(feature_name):
    if "Region_Code" in feature_name:
        return "Region_Code"
    if "Policy_Sales_Channel" in feature_name:
        return "Policy_Sales_Channel"
    if "Gender" in feature_name:
        return "Gender"
    if "Vehicle_Age" in feature_name:
        return "Vehicle_Age"
    if "Vehicle_Damage" in feature_name:
        return "Vehicle_Damage"
    if "numerical__Age" in feature_name:
        return "Age"
    if "numerical__Annual_Premium" in feature_name:
        return "Annual_Premium"
    if "numerical__Vintage" in feature_name:
        return "Vintage"
    if "binary__Driving_License" in feature_name:
        return "Driving_License"
    if "binary__Previously_Insured" in feature_name:
        return "Previously_Insured"
    return feature_name

permutation_importance_df["feature_group"] = (
    permutation_importance_df["feature"]
    .map(assign_feature_group)
)

permutation_importance_top_features = (
    permutation_importance_df
    .sort_values("importance_mean", ascending=False)
    .head(25)
)

grouped_importance_df = (
    permutation_importance_df
    .groupby("feature_group", as_index=False)
    .agg(
        importance_mean=("importance_mean", "sum"),
        importance_std=("importance_std", "sum"),
        encoded_feature_count=("feature", "count"),
    )
    .sort_values("importance_mean", ascending=False)
)

display(
    permutation_importance_top_features.style.format({
        "importance_mean": "{:.6f}",
        "importance_std": "{:.6f}",
    })
)

display(
    grouped_importance_df.style.format({
        "importance_mean": "{:.6f}",
        "importance_std": "{:.6f}",
    })
)

,feature,importance_mean,importance_std,feature_group
218,binary__Previously_Insured,0.108468,0.004709,Previously_Insured
214,numerical__Age,0.061564,0.002071,Age
5,categorical__Vehicle_Damage_No,0.053954,0.004983,Vehicle_Damage
204,categorical__Policy_Sales_Channel_152.0,0.031803,0.003067,Policy_Sales_Channel
3,categorical__Vehicle_Age_< 1 Year,0.020390,0.003703,Vehicle_Age
84,categorical__Policy_Sales_Channel_26.0,0.014765,0.001482,Policy_Sales_Channel
6,categorical__Vehicle_Damage_Yes,0.013936,0.003862,Vehicle_Damage
212,categorical__Policy_Sales_Channel_160.0,0.008601,0.001088,Policy_Sales_Channel
2,categorical__Vehicle_Age_1-2 Year,0.008235,0.001146,Vehicle_Age
215,numerical__Annual_Premium,0.007198,0.002195,Annual_Premium


,feature_group,importance_mean,importance_std,encoded_feature_count
5,Previously_Insured,0.108468,0.004709,1
8,Vehicle_Damage,0.067891,0.008845,2
4,Policy_Sales_Channel,0.063411,0.012268,154
0,Age,0.061564,0.002071,1
7,Vehicle_Age,0.031185,0.005094,3
6,Region_Code,0.017049,0.014454,53
1,Annual_Premium,0.007198,0.002195,1
3,Gender,0.001313,0.000570,2
9,Vintage,0.000848,0.001212,1
2,Driving_License,0.000141,0.000319,1


### 8.7 Leave-one-feature-out ablations

This focused pass removes one remaining raw feature at a time from the all-features reference configuration. It uses the same tuned weighted LightGBM, preprocessing protocol, validation split, and top-K metrics. The final test set is not used.

In [54]:
remaining_features_for_ablation = [
    "Gender",
    "Vehicle_Age",
    "Vehicle_Damage",
    "Age",
    "Annual_Premium",
    "Vintage",
    "Driving_License",
    "Previously_Insured",
]

leave_one_out_results = []

for feature_name in remaining_features_for_ablation:
    leave_one_out_results.append(
        evaluate_feature_variant(
            variant_name=f"without_{feature_name}",
            train_data=X_train,
            validation_data=X_val,
            dropped_features=[feature_name],
        )
    )

leave_one_out_results_df = pd.DataFrame(leave_one_out_results)
leave_one_out_results_df["delta_pr_auc"] = (
    leave_one_out_results_df["pr_auc"] - baseline_pr_auc
)
leave_one_out_results_df["delta_top10_lift"] = (
    leave_one_out_results_df["lift_at_10"] - baseline_top10_lift
)

leave_one_out_results_df = leave_one_out_results_df.sort_values(
    "delta_pr_auc",
    ascending=False,
).reset_index(drop=True)

display(leave_one_out_results_df.style.format({
    "pr_auc": "{:.6f}",
    "roc_auc": "{:.6f}",
    "precision_at_10": "{:.4%}",
    "recall_at_10": "{:.4%}",
    "lift_at_10": "{:.6f}",
    "delta_pr_auc": "{:+.6f}",
    "delta_top10_lift": "{:+.6f}",
}))

,variant,pr_auc,roc_auc,precision_at_10,recall_at_10,lift_at_10,delta_pr_auc,delta_top10_lift
0,without_Vintage,0.370049,0.857751,39.5959%,32.3057%,3.230656,+0.000042,-0.004282
1,without_Driving_License,0.369455,0.857155,39.5434%,32.2629%,3.226375,-0.000552,-0.008564
2,without_Gender,0.369187,0.857102,39.5959%,32.3057%,3.230656,-0.000820,-0.004282
3,without_Annual_Premium,0.367809,0.856890,39.4385%,32.1773%,3.217811,-0.002197,-0.017127
4,without_Vehicle_Age,0.362196,0.855994,38.9399%,31.7705%,3.177133,-0.007811,-0.057805
5,without_Previously_Insured,0.357569,0.847138,38.6513%,31.5350%,3.153583,-0.012437,-0.081355
6,without_Vehicle_Damage,0.351265,0.845626,37.5230%,30.6144%,3.061523,-0.018742,-0.173415
7,without_Age,0.334846,0.842139,34.6103%,28.2381%,2.823881,-0.035160,-0.411058


### 8.8 Analysis of permutation importance and leave-one-feature-out results

The all-features configuration remains the reference: PR-AUC `0.370007`, ROC-AUC `0.857251`, Precision@10 `39.6484%`, Recall@10 `32.3485%`, and Lift@10 `3.234938`.

#### Leave-one-feature-out findings

- `Age` is the most consequential remaining feature. Removing it reduced PR-AUC by `0.035160` and top-10% Lift by `0.411057`.
- `Vehicle_Damage` is the next strongest feature. Removing it reduced PR-AUC by `0.018742` and top-10% Lift by `0.173415`.
- `Previously_Insured` is also highly valuable. Removing it reduced PR-AUC by `0.012437` and top-10% Lift by `0.081355`.
- `Vehicle_Age` provides meaningful but smaller signal, with a PR-AUC decrease of `0.007811` after removal.
- `Annual_Premium` contributes modestly: removing it reduced PR-AUC by `0.002197`.
- `Gender`, `Driving_License`, and `Vintage` have little measurable incremental value in this validation experiment. Removing `Vintage` slightly increased PR-AUC by `0.000042`, but reduced top-10% Lift by `0.004282`; this is too small and inconsistent to justify removing it.

#### Permutation-importance findings

The grouped permutation results support the leave-one-out conclusions. The largest grouped importance values were `Previously_Insured` (`0.108468`), `Vehicle_Damage` (`0.067891`), `Policy_Sales_Channel` (`0.063411`), `Age` (`0.061564`), and `Vehicle_Age` (`0.031185`). `Region_Code` had lower grouped permutation importance (`0.017049`) but the earlier group ablation showed that removing it still causes a material validation decline.

The exact numerical ranking between permutation importance and ablation should not be treated as absolute. Permutation importance shuffles transformed one-hot columns individually and then aggregates them, while ablation retrains the model after removing a raw feature. Correlated categories and feature interactions can therefore make the two methods produce different magnitudes. The ablation results are the stronger evidence for deciding whether a raw feature group should be retained.

#### Feature-set decision

No remaining feature should be removed solely from these results. The small changes for `Gender`, `Driving_License`, and `Vintage` are within the range where split-specific variation may dominate, and all-features remains the safest reference configuration. The current evidence supports retaining the existing feature set and stopping broad single-feature ablation work.



## 9. Stability and uncertainty analysis

The next two experiments measure how dependable the provisional weighted LightGBM result is. Repeated stratified cross-validation measures variation caused by different training/validation partitions. Bootstrap resampling estimates uncertainty around the metrics on the current validation split. Neither experiment uses the untouched test set.

### 9.1 Repeated stratified cross-validation stability

The preprocessing step is cloned and fitted inside every fold so that each validation fold is scored only with transformations learned from that fold's training portion. The tuned weighted LightGBM hyperparameters are held fixed; this experiment measures variability, not further tuning.

In [55]:
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline

selected_lgbm = tuned_models["class_weighted"]
repeated_cv = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=3,
    random_state=random_state,
)

repeated_cv_rows = []
for fold_id, (fit_idx, validation_idx) in enumerate(
    repeated_cv.split(X_train, y_train),
    start=1,
):
    X_fold_train = X_train.iloc[fit_idx]
    X_fold_validation = X_train.iloc[validation_idx]
    y_fold_train = y_train.iloc[fit_idx]
    y_fold_validation = y_train.iloc[validation_idx]

    fold_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", clone(selected_lgbm)),
    ])
    fold_pipeline.fit(X_fold_train, y_fold_train)
    fold_probabilities = fold_pipeline.predict_proba(X_fold_validation)[:, 1]

    for selection_percentage in top_k_percentages:
        repeated_cv_rows.append({
            "fold": fold_id,
            "selection_percentage": selection_percentage,
            "pr_auc": float(pr_auc(y_fold_validation, fold_probabilities)),
            "roc_auc": float(roc_auc_score(y_fold_validation, fold_probabilities)),
            "precision_at_k": float(precision_at_k(y_fold_validation, fold_probabilities, selection_percentage)),
            "recall_at_k": float(recall_at_k(y_fold_validation, fold_probabilities, selection_percentage)),
            "lift_at_k": float(Lift_at_k(y_fold_validation, fold_probabilities, selection_percentage)),
        })

repeated_cv_results = pd.DataFrame(repeated_cv_rows)
repeated_cv_summary = (
    repeated_cv_results
    .groupby("selection_percentage")
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        precision_at_k_mean=("precision_at_k", "mean"),
        precision_at_k_std=("precision_at_k", "std"),
        recall_at_k_mean=("recall_at_k", "mean"),
        recall_at_k_std=("recall_at_k", "std"),
        lift_at_k_mean=("lift_at_k", "mean"),
        lift_at_k_std=("lift_at_k", "std"),
    )
    .reset_index()
)

display(repeated_cv_results.style.format({
    "pr_auc": "{:.6f}",
    "roc_auc": "{:.6f}",
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.6f}",
}))
display(repeated_cv_summary.style.format({
    "pr_auc_mean": "{:.6f}", "pr_auc_std": "{:.6f}",
    "roc_auc_mean": "{:.6f}", "roc_auc_std": "{:.6f}",
    "precision_at_k_mean": "{:.4%}", "precision_at_k_std": "{:.4%}",
    "recall_at_k_mean": "{:.4%}", "recall_at_k_std": "{:.4%}",
    "lift_at_k_mean": "{:.6f}", "lift_at_k_std": "{:.6f}",
}))

if mlflow.active_run() is not None:
    mlflow.end_run(status="KILLED")
with mlflow.start_run(run_name="tier2_repeated_cv_stability"):
    mlflow.log_params({
        "model_variant": "lightgbm_class_weighted",
        "cv_type": "repeated_stratified",
        "n_splits": 3,
        "n_repeats": 3,
        "selection_percentages": ",".join(map(str, top_k_percentages)),
    })
    top10_summary = repeated_cv_summary.query("selection_percentage == 10").iloc[0]
    mlflow.log_metrics({
        "stability_pr_auc_mean": float(top10_summary["pr_auc_mean"]),
        "stability_pr_auc_std": float(top10_summary["pr_auc_std"]),
        "stability_precision_at_10_mean": float(top10_summary["precision_at_k_mean"]),
        "stability_precision_at_10_std": float(top10_summary["precision_at_k_std"]),
        "stability_recall_at_10_mean": float(top10_summary["recall_at_k_mean"]),
        "stability_recall_at_10_std": float(top10_summary["recall_at_k_std"]),
        "stability_lift_at_10_mean": float(top10_summary["lift_at_k_mean"]),
        "stability_lift_at_10_std": float(top10_summary["lift_at_k_std"]),
    })
    mlflow.log_text(repeated_cv_results.to_csv(index=False), "repeated_cv_results.csv")
    mlflow.log_text(repeated_cv_summary.to_csv(index=False), "repeated_cv_summary.csv")

,fold,selection_percentage,pr_auc,roc_auc,precision_at_k,recall_at_k,lift_at_k
0,1,1,0.372008,0.859353,48.3268%,3.9419%,3.943001
1,1,5,0.372008,0.859353,43.4560%,17.7264%,3.545593
2,1,10,0.372008,0.859353,39.8642%,32.5225%,3.252536
3,1,20,0.372008,0.859353,35.9016%,58.5822%,2.929226
4,2,1,0.368874,0.858001,46.3583%,3.7813%,3.782390
5,2,5,0.368874,0.858001,42.3342%,17.2688%,3.454063
6,2,10,0.368874,0.858001,39.9429%,32.5867%,3.258959
7,2,20,0.368874,0.858001,35.6851%,58.2290%,2.911563
8,3,1,0.364960,0.855129,46.1614%,3.7653%,3.766328
9,3,5,0.364960,0.855129,42.3735%,17.2848%,3.457275


,selection_percentage,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,precision_at_k_mean,precision_at_k_std,recall_at_k_mean,recall_at_k_std,lift_at_k_mean,lift_at_k_std
0,1,0.369748,0.002800,0.857603,0.001634,47.3863%,1.1458%,3.8652%,0.0935%,3.866264,0.093489
1,5,0.369748,0.002800,0.857603,0.001634,42.8415%,0.4552%,17.4757%,0.1857%,3.495457,0.037140
2,10,0.369748,0.002800,0.857603,0.001634,39.7374%,0.3405%,32.4190%,0.2778%,3.242187,0.027783
3,20,0.369748,0.002800,0.857603,0.001634,35.6255%,0.2662%,58.1317%,0.4343%,2.906701,0.021717


🏃 View run tier2_repeated_cv_stability at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/05b75ace8cbc41779b7330487c3aebec
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


### 9.2 Bootstrap confidence intervals on validation metrics

This resamples the existing validation predictions to estimate a 95% confidence interval for PR-AUC, ROC-AUC, and top-10% selection metrics. It is an uncertainty estimate for validation evidence, not a replacement for the final one-time evaluation on the untouched test set.

In [56]:
bootstrap_rng = np.random.default_rng(random_state)
n_bootstrap = 1000
top10_percentage = 10
validation_probabilities = selected_lgbm.predict_proba(X_val_processed)[:, 1]
validation_labels = np.asarray(y_val)

bootstrap_rows = []
for iteration in range(n_bootstrap):
    sample_indices = bootstrap_rng.integers(
        0,
        len(validation_labels),
        size=len(validation_labels),
    )
    sampled_labels = validation_labels[sample_indices]
    sampled_probabilities = validation_probabilities[sample_indices]

    if np.unique(sampled_labels).size < 2:
        continue

    bootstrap_rows.append({
        "iteration": iteration + 1,
        "pr_auc": float(pr_auc(sampled_labels, sampled_probabilities)),
        "roc_auc": float(roc_auc_score(sampled_labels, sampled_probabilities)),
        "precision_at_10": float(precision_at_k(sampled_labels, sampled_probabilities, top10_percentage)),
        "recall_at_10": float(recall_at_k(sampled_labels, sampled_probabilities, top10_percentage)),
        "lift_at_10": float(Lift_at_k(sampled_labels, sampled_probabilities, top10_percentage)),
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_summary = pd.DataFrame({
    "metric": bootstrap_results.columns[1:],
    "mean": [bootstrap_results[metric].mean() for metric in bootstrap_results.columns[1:]],
    "std": [bootstrap_results[metric].std() for metric in bootstrap_results.columns[1:]],
    "lower_2_5": [bootstrap_results[metric].quantile(0.025) for metric in bootstrap_results.columns[1:]],
    "upper_97_5": [bootstrap_results[metric].quantile(0.975) for metric in bootstrap_results.columns[1:]],
})

print(f"Valid bootstrap samples: {len(bootstrap_results):,} / {n_bootstrap:,}")
display(bootstrap_summary.style.format({
    "mean": "{:.6f}",
    "std": "{:.6f}",
    "lower_2_5": "{:.6f}",
    "upper_97_5": "{:.6f}",
}))

if mlflow.active_run() is not None:
    mlflow.end_run(status="KILLED")
with mlflow.start_run(run_name="tier2_bootstrap_validation_uncertainty"):
    mlflow.log_params({
        "model_variant": "lightgbm_class_weighted",
        "resampling": "validation_bootstrap",
        "n_bootstrap": n_bootstrap,
        "confidence_interval": "95%",
        "selection_percentage": top10_percentage,
    })
    bootstrap_metrics = {}
    for _, summary_row in bootstrap_summary.iterrows():
        metric_name = summary_row["metric"]
        safe_metric_name = metric_name.replace("_", "_")
        bootstrap_metrics[f"bootstrap_{safe_metric_name}_mean"] = float(summary_row["mean"])
        bootstrap_metrics[f"bootstrap_{safe_metric_name}_std"] = float(summary_row["std"])
        bootstrap_metrics[f"bootstrap_{safe_metric_name}_lower_2_5"] = float(summary_row["lower_2_5"])
        bootstrap_metrics[f"bootstrap_{safe_metric_name}_upper_97_5"] = float(summary_row["upper_97_5"])
    mlflow.log_metrics(bootstrap_metrics)
    mlflow.log_text(bootstrap_results.to_csv(index=False), "bootstrap_validation_results.csv")
    mlflow.log_text(bootstrap_summary.to_csv(index=False), "bootstrap_validation_summary.csv")

Valid bootstrap samples: 1,000 / 1,000


,metric,mean,std,lower_2_5,upper_97_5
0,pr_auc,0.370264,0.006690,0.356191,0.383299
1,roc_auc,0.857218,0.002141,0.852986,0.861521
2,precision_at_10,0.396106,0.008096,0.379684,0.412753
3,recall_at_10,0.323327,0.005763,0.312107,0.334824
4,lift_at_10,3.233358,0.057628,3.121153,3.348328


🏃 View run tier2_bootstrap_validation_uncertainty at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/a18a33bdf5ec4f99af36437078ec8480
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


### 9.3 Stability-analysis conclusion

The provisional class-weighted LightGBM is stable across the repeated training-only CV partitions. Across 9 folds, mean PR-AUC was `0.369748 ± 0.002800`, mean ROC-AUC was `0.857603 ± 0.001634`, and top-10% Lift was `3.242 ± 0.028`. The 1,000 validation bootstrap samples produced a top-10% Lift of `3.233` with a 95% interval of `3.121–3.348`.

These results support proceeding to operating-capacity confirmation and model freeze. They do not establish that top 10% is universally correct for every company; the company must approve that capacity as a pilot operating rule. The untouched test set remains reserved for the single final evaluation after the candidate and operating point are frozen.

## 10. Final model freeze and one-time test evaluation

The stability analysis supports moving from exploration to a controlled final evaluation. This section records the approved weighted LightGBM configuration, confirms the top-10% validation operating rule, refits the frozen pipeline on the approved training data (`train + validation`), and evaluates the untouched test set exactly once. The capacity remains configurable in the specification even though 10% is the current Phase 1 baseline.

### 10.1 Freeze decision record and validation operating point

The model family, features, preprocessing design, class-weighting method, split protocol, and ranking rule are fixed here before the final test evaluation. The validation set is used to confirm the top-10% operating point; it is not used to change the model after this record is created.

In [57]:
import json
from sklearn.base import clone

selected_lgbm = tuned_models["class_weighted"]
approved_selection_percentage = 10
approved_selection_rule = (
    f"Rank customers by predicted response propensity and select the top {approved_selection_percentage}%"
)

# This is the MLflow run that produced the currently approved weighted tuning result.
source_tuning_run_id = "e573ee5423434e6485404a96ca5f5731"
approved_raw_features = [str(column) for column in X_train.columns]
approved_transformed_features = [str(column) for column in feature_names]
approved_model_params = {
    str(parameter): str(value)
    for parameter, value in selected_lgbm.get_params().items()
}
approved_preprocessor_params = {
    str(parameter): str(value)
    for parameter, value in preprocessor.get_params(deep=True).items()
}
approved_scale_pos_weight = float(selected_lgbm.get_params()["scale_pos_weight"])

validation_probabilities_for_freeze = selected_lgbm.predict_proba(X_val_processed)[:, 1]
validation_operating_rows = []
for selection_percentage in top_k_percentages:
    validation_operating_rows.append({
        "selection_percentage": selection_percentage,
        "pr_auc": float(pr_auc(y_val, validation_probabilities_for_freeze)),
        "roc_auc": float(roc_auc_score(y_val, validation_probabilities_for_freeze)),
        "precision_at_k": float(precision_at_k(y_val, validation_probabilities_for_freeze, selection_percentage)),
        "recall_at_k": float(recall_at_k(y_val, validation_probabilities_for_freeze, selection_percentage)),
        "lift_at_k": float(Lift_at_k(y_val, validation_probabilities_for_freeze, selection_percentage)),
    })
validation_operating_results = pd.DataFrame(validation_operating_rows)
selected_validation_metrics = (
    validation_operating_results
    .loc[validation_operating_results["selection_percentage"] == approved_selection_percentage]
    .iloc[0]
)

model_freeze_spec = {
    "model_family": "LightGBM",
    "model_variant": "class_weighted",
    "source_tuning_run_id": source_tuning_run_id,
    "exact_model_parameters": approved_model_params,
    "class_weighting": "scale_pos_weight",
    "scale_pos_weight": approved_scale_pos_weight,
    "raw_feature_list": approved_raw_features,
    "transformed_feature_count": len(approved_transformed_features),
    "preprocessing": "ColumnTransformer from the approved notebook preprocessing flow",
    "preprocessor_parameters": approved_preprocessor_params,
    "split_protocol": "stratified 80% train / 10% validation / 10% test, random_state=42",
    "training_for_final_fit": "train + validation after candidate and operating-point decisions",
    "operating_capacity_percentage": approved_selection_percentage,
    "operating_capacity_configurable": True,
    "ranking_rule": approved_selection_rule,
    "validation_metrics_at_approved_capacity": {
        str(metric): float(selected_validation_metrics[metric])
        for metric in ["pr_auc", "roc_auc", "precision_at_k", "recall_at_k", "lift_at_k"]
    },
    "test_evaluation_status": "not_yet_run",
}

display(validation_operating_results.style.format({
    "pr_auc": "{:.6f}",
    "roc_auc": "{:.6f}",
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.6f}",
}))
print(json.dumps(model_freeze_spec, indent=2))

,selection_percentage,pr_auc,roc_auc,precision_at_k,recall_at_k,lift_at_k
0,1,0.370007,0.857251,50.6562%,4.1319%,4.133070
1,5,0.370007,0.857251,41.4698%,16.9129%,3.383550
2,10,0.370007,0.857251,39.6484%,32.3485%,3.234938
3,20,0.370007,0.857251,35.5025%,57.9319%,2.896672


{
  "model_family": "LightGBM",
  "model_variant": "class_weighted",
  "source_tuning_run_id": "e573ee5423434e6485404a96ca5f5731",
  "exact_model_parameters": {
    "boosting_type": "gbdt",
    "class_weight": "None",
    "colsample_bytree": "0.8",
    "importance_type": "split",
    "learning_rate": "0.03",
    "max_depth": "-1",
    "min_child_samples": "100",
    "min_child_weight": "0.001",
    "min_split_gain": "0.0",
    "n_estimators": "700",
    "n_jobs": "-1",
    "num_leaves": "31",
    "objective": "binary",
    "random_state": "42",
    "reg_alpha": "1.0",
    "reg_lambda": "1.0",
    "subsample": "0.8",
    "subsample_for_bin": "200000",
    "subsample_freq": "1",
    "verbosity": "-1",
    "scale_pos_weight": "7.159039820166988"
  },
  "class_weighting": "scale_pos_weight",
  "scale_pos_weight": 7.159039820166988,
  "raw_feature_list": [
    "Gender",
    "Age",
    "Driving_License",
    "Region_Code",
    "Previously_Insured",
    "Vehicle_Age",
    "Vehicle_Damage",
  

### 10.2 Train the frozen pipeline on the approved training data

The candidate and operating point have already been selected using the original training/validation protocol. The final artifact is now refit on the combined training and validation rows so it can learn from all data available before the untouched test set. The preprocessing is cloned and refitted on this combined data, then stored together with the model as one pipeline.

In [59]:
from sklearn.pipeline import Pipeline

X_final_train = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_final_train = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

final_preprocessor = clone(preprocessor)
final_model = clone(selected_lgbm)
final_pipeline = Pipeline(steps=[
    ("preprocessor", final_preprocessor),
    ("model", final_model),
])
final_pipeline.fit(X_final_train, y_final_train)

final_transformed_features = [
    str(column)
    for column in final_pipeline.named_steps["preprocessor"].get_feature_names_out()
]

# Refitting on train + validation can add categories that were absent from train alone.
# That changes the one-hot feature list legitimately; it does not indicate a model error.
raw_final_features = [str(column) for column in X_final_train.columns]
assert raw_final_features == approved_raw_features
assert final_pipeline.named_steps["model"].n_features_in_ == len(final_transformed_features)
new_transformed_features = sorted(set(final_transformed_features) - set(approved_transformed_features))
missing_transformed_features = sorted(set(approved_transformed_features) - set(final_transformed_features))

model_freeze_spec["final_transformed_feature_list"] = final_transformed_features
model_freeze_spec["final_transformed_feature_count"] = len(final_transformed_features)

print(f"New one-hot columns after adding validation rows: {len(new_transformed_features):,}")
print(f"Approved train-only columns absent from final fit: {len(missing_transformed_features):,}")

print(f"Final training rows: {len(X_final_train):,}")
print(f"Final training positive responses: {int(y_final_train.sum()):,}")
print(f"Final transformed feature count: {len(final_transformed_features):,}")
print("Final pipeline contains the approved preprocessor and weighted LightGBM model.")

New one-hot columns after adding validation rows: 1
Approved train-only columns absent from final fit: 0
Final training rows: 342,998
Final training positive responses: 42,039
Final transformed feature count: 220
Final pipeline contains the approved preprocessor and weighted LightGBM model.


### 10.3 One-time untouched-test evaluation and MLflow logging

This cell makes the only final test prediction for the frozen candidate. Test results are reported for the complete ranking and for the configurable top-K capacities, with top 10% treated as the current Phase 1 operating point. The test result must not be used to change the model or capacity afterward.

In [60]:
import mlflow.sklearn

if mlflow.active_run() is not None:
    mlflow.end_run(status="KILLED")

with mlflow.start_run(run_name="final_frozen_lgbm_top10") as final_run:
    final_run_id = final_run.info.run_id
    frozen_model_uri = f"runs:/{final_run_id}/model"

    mlflow.set_tags({
        "stage": "final_frozen_candidate",
        "model_family": "LightGBM",
        "model_variant": "class_weighted",
        "test_evaluation": "one_time_final_evaluation",
        "operating_capacity": "top_10_percent_baseline_configurable",
    })
    mlflow.log_params({
        "model_family": "LightGBM",
        "model_variant": "class_weighted",
        "class_weighting": "scale_pos_weight",
        "scale_pos_weight": str(approved_scale_pos_weight),
        "split_protocol": "stratified_80_10_10_random_state_42",
        "final_fit_data": "train_plus_validation",
        "operating_capacity_percentage": str(approved_selection_percentage),
        "capacity_configurable": "true",
        "ranking_rule": "highest_predicted_response_scores",
        "source_tuning_run_id": source_tuning_run_id,
    })
    mlflow.log_params({
        f"model_{parameter}": value
        for parameter, value in approved_model_params.items()
    })
    mlflow.log_text(
        json.dumps(model_freeze_spec, indent=2),
        artifact_file="model_freeze_spec.json",
    )
    mlflow.log_text(
        json.dumps({"raw_features": approved_raw_features, "transformed_features": final_transformed_features}, indent=2),
        artifact_file="feature_list.json",
    )
    mlflow.sklearn.log_model(final_pipeline, artifact_path="model")

    # This is the only final test prediction call in the freeze workflow.
    final_test_probabilities = final_pipeline.predict_proba(X_test)[:, 1]
    final_test_rows = []
    for selection_percentage in top_k_percentages:
        final_test_rows.append({
            "selection_percentage": selection_percentage,
            "pr_auc": float(pr_auc(y_test, final_test_probabilities)),
            "roc_auc": float(roc_auc_score(y_test, final_test_probabilities)),
            "precision_at_k": float(precision_at_k(y_test, final_test_probabilities, selection_percentage)),
            "recall_at_k": float(recall_at_k(y_test, final_test_probabilities, selection_percentage)),
            "lift_at_k": float(Lift_at_k(y_test, final_test_probabilities, selection_percentage)),
        })
    final_test_metrics = pd.DataFrame(final_test_rows)
    selected_test_metrics = (
        final_test_metrics
        .loc[final_test_metrics["selection_percentage"] == approved_selection_percentage]
        .iloc[0]
    )

    mlflow.log_metrics({
        "test_pr_auc": float(selected_test_metrics["pr_auc"]),
        "test_roc_auc": float(selected_test_metrics["roc_auc"]),
        "test_precision_at_10pct": float(selected_test_metrics["precision_at_k"]),
        "test_recall_at_10pct": float(selected_test_metrics["recall_at_k"]),
        "test_lift_at_10pct": float(selected_test_metrics["lift_at_k"]),
    })
    mlflow.log_text(
        final_test_metrics.to_csv(index=False),
        artifact_file="final_test_metrics.csv",
    )
    final_spec_with_result = {
        **model_freeze_spec,
        "test_evaluation_status": "completed_once",
        "final_mlflow_run_id": final_run_id,
        "frozen_model_uri": frozen_model_uri,
        "final_test_metrics_at_top_10_percent": {
            str(metric): float(selected_test_metrics[metric])
            for metric in ["pr_auc", "roc_auc", "precision_at_k", "recall_at_k", "lift_at_k"]
        },
    }
    mlflow.log_text(
        json.dumps(final_spec_with_result, indent=2),
        artifact_file="final_model_record.json",
    )

display(final_test_metrics.style.format({
    "pr_auc": "{:.6f}",
    "roc_auc": "{:.6f}",
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.6f}",
}))
print(f"Final MLflow run ID: {final_run_id}")
print(f"Frozen model URI: {frozen_model_uri}")
print("The final test result is recorded for reporting; do not retune the model from it.")

2026/08/23 17:09:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/23 17:09:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/23 17:09:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run final_frozen_lgbm_top10 at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2/runs/dc28826afdcc4e5cbce068db62439094
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/2


,selection_percentage,pr_auc,roc_auc,precision_at_k,recall_at_k,lift_at_k
0,1,0.373937,0.859726,50.6562%,4.1319%,4.133070
1,5,0.373937,0.859726,41.7323%,17.0199%,3.404965
2,10,0.373937,0.859726,39.6221%,32.3271%,3.232797
3,20,0.373937,0.859726,35.8961%,58.5742%,2.928786


Final MLflow run ID: dc28826afdcc4e5cbce068db62439094
Frozen model URI: runs:/dc28826afdcc4e5cbce068db62439094/model
The final test result is recorded for reporting; do not retune the model from it.


### 10.4 Final frozen-model test analysis and decision

The final frozen weighted LightGBM was refit on the combined training and validation data and evaluated once on the untouched test set. The final MLflow run is `dc28826afdcc4e5cbce068db62439094`, and the logged model URI is `runs:/dc28826afdcc4e5cbce068db62439094/model`.

#### Final test results

- PR-AUC: `0.373937`
- ROC-AUC: `0.859726`
- At top 1%: Precision `50.6562%`, Recall `4.1319%`, Lift `4.133070`
- At top 5%: Precision `41.7323%`, Recall `17.0199%`, Lift `3.404965`
- At the Phase 1 top-10% capacity: Precision `39.6221%`, Recall `32.3271%`, Lift `3.232797`
- At top 20%: Precision `35.8961%`, Recall `58.5742%`, Lift `2.928786`

#### What the results mean

The top-10% result means that contacting the 10% highest-scoring customers identifies about `32.33%` of all positive responders, while the selected group responds at roughly `3.23×` the overall test-set response rate. This supports the project objective of prioritizing outreach, not making claims, pricing, eligibility, or underwriting decisions.

The test PR-AUC (`0.373937`) and ROC-AUC (`0.859726`) are slightly higher than the earlier validation reference (`0.370007` PR-AUC and `0.857251` ROC-AUC). More importantly, top-10% Precision (`39.6221%`), Recall (`32.3271%`), and Lift (`3.232797`) are almost identical to validation (`39.6484%`, `32.3485%`, and `3.234938`). This close alignment is evidence that the model did not materially overfit the validation split.

The final test values also fall inside the validation bootstrap intervals recorded in Section 9.2. The test Lift is within the earlier interval of `3.121–3.348`, and the test PR-AUC is within `0.356191–0.383299`. This is consistent with the repeated-CV finding that the candidate is stable.

#### Frozen-model decision

The weighted LightGBM, its preprocessing pipeline, feature set, class-weighting configuration, split protocol, top-10% baseline capacity, and ranking rule are now frozen for this experiment. The test result is a final reportable estimate and must not be used to retune the model. The top-10% capacity remains configurable for future companies or campaigns.

The MLflow warnings about deprecated `artifact_path` usage and pickle/cloudpickle safety did not invalidate the run or metrics. They should be addressed during production packaging by using the current MLflow model-logging API and choosing a reviewed model serialization strategy.